# Optical or Demographic? A Confound-Controlled Benchmark and Null-Calibrated Explainability Audit for Four-Wavelength Reflectance PPG Haemoglobin Estimation

**Author:** Fahad Ibne Firoz  
**Artefact type:** computational study, notebook only. No physical sensor, no embedded deployment, no clinical validation is claimed anywhere.

---

## 1. What this study asks

Non-invasive haemoglobin (Hb) estimation from photoplethysmography is a crowded field reporting increasingly strong numbers. Almost every published pipeline feeds **demographic variables (age, sex, height, weight) into the regressor alongside the optical features**, and almost none reports the demographic-only control that would show how much the optics contributed. Because age and sex are themselves strong physiological predictors of haemoglobin, a model that reports *r* = 0.64 with demographics in the feature vector has not demonstrated that it read any light.

This notebook asks four questions that the field has left open:

- **RQ1.** On a public four-wavelength reflectance PPG dataset, how much predictive information about haemoglobin is carried by the *optics*, over and above what demographics alone provide?
- **RQ2.** Do Beer-Lambert-grounded features (AC/DC, ratio-of-ratios) recover optical information that generic waveform statistics miss? Does a learned 1D-CNN representation recover more than either?
- **RQ3.** When SHAP ranks wavelengths or features, is that ranking distinguishable from the ranking a model produces when it has learned *nothing*?
- **RQ4.** Does apparent wavelength importance track haemoglobin optics, or does it track the per-channel photon budget (signal-to-noise ratio)?

---

## 2. Gaps this study closes

| # | Gap in the current literature | How it is closed here |
|---|---|---|
| G1 | Demographic contribution is never isolated; optical credit is assumed | Pre-registered three-arm design with a **permutation null on the optical block** |
| G2 | Feature construction is ad hoc; the Beer-Lambert quantity (AC/DC) is often absent or destroyed by detrending | Two feature banks run head-to-head on **identical folds**, plus a learned representation arm |
| G3 | XAI is reported as evidence of mechanism, with no test of whether the attribution is meaningful | **Null-calibrated attribution audit**: the same SHAP pipeline run against a label-permuted model |
| G4 | Channel-reduction claims assume importance reflects physiology | **Photon-budget confound test**: importance regressed on measured per-channel SNR |
| G5 | Non-significance is read as equivalence | Pre-registered **non-inferiority margin** with bootstrap CI on the paired difference |

---

## 3. Contributions

1. **Optical Information Gain (OIG)** — a confound-controlled statistic with a permutation null that quantifies what the optics add over demographics. Reported with bootstrap CI and an exact permutation p-value. This is the headline result and it is publishable whichever way it comes out.
2. **A three-arm representation comparison** — Beer-Lambert physics features, generic waveform statistics, and a GPU-trained 1D-CNN on raw four-channel PPG — evaluated on one frozen subject-level fold structure.
3. **The Null-Calibrated Attribution Audit (NCAA)** and its summary statistic, the **Attribution Credibility Index (ACI)**: the rank agreement between SHAP and retrain-and-remove importance, expressed as a z-score against the distribution of that same agreement under a label-permuted null. The claim under test is that SHAP cannot signal that a model has learned nothing.
4. **The photon-budget confound test** — measured per-channel SNR as a competing explanation for wavelength importance, plus an SNR-stratified re-run of the full ablation.
5. **A cross-target specificity control** — the identical pipeline applied to fasting glucose and brachial blood pressure, which are recorded in the same dataset. This separates "these features are inert" from "this pipeline is broken", a positive control that single-target studies cannot provide.

---

## 4. Dataset

**Hb-PPG** (Chen et al., *Scientific Data*, 2026, DOI `10.1038/s41597-026-06945-6`; Figshare `10.6084/m9.figshare.22256143`).

- 252 adult subjects, aged 21-90 (mean 47 ± 21), 109 male / 143 female (56.7 % female)
- Four-wavelength **fingertip reflectance** PPG: 660, 730, 850, 940 nm
- 200 Hz, 12-bit ADC, 60 s nominal (the workbook records 60 / 50 / 45 s variants)
- CSV per subject, wavelengths in **columns 1-4 in the order above**; `.mat` mirrors available
- `subject information.xlsx`: ID, age, gender, height, weight, Hb, fasting glucose, SBP, DBP, signal length
- Hb reference: HemoCue Hb 201+, calibrated against the ICSH HiCN method. Range 85-173 g/L
- Sensor: ADPD4100 AFE + DCM08 PPG front end + FSR400 contact-pressure sensor
- Dataset authors report per-channel SNR: 850 nm cleanest (19.04 dB, SD 4.77), 940 nm noisiest (16.44 dB, SD 6.53, 17.5 % of subjects below 10 dB)
- The dataset paper explicitly leaves downstream modelling to future work, so there is **no published modelling baseline on this dataset** to compare against

---

## 5. Pipeline, in words

No code in this section. This is the map.

**Stage A — Environment and data.** Detect hardware, install missing packages, locate the workbook and the 252 signal CSVs, assert the four-column wavelength order rather than assuming it, load metadata, convert Hb from g/L to g/dL, treat placeholder strings as missing rather than zero.

**Stage B — Signal conditioning (the correction).** For each channel, interpolate non-finite samples and then produce **three distinct arrays**: the untouched raw signal, a linearly detrended copy, and a 0.5-8 Hz band-passed copy. The original pipeline reassigned the detrended array back onto the variable it then returned as "raw", which silently forced every DC-dependent quantity to zero. Keeping the three arrays separate is what makes the Beer-Lambert features recoverable.

**Stage C — Signal quality.** Per subject and per channel, compute SNR as the band-pass to band-stop power ratio in dB, matching the definition the dataset authors used, so our numbers are comparable to theirs. These are quality-control variables and are **never** admitted to the predictor matrix.

**Stage D — Three feature banks, kept strictly apart.**
- *Physics bank*: DC from the raw signal, AC from the band-passed signal, AC/DC per channel, perfusion index, log-DC, and all six cross-wavelength ratio-of-ratios plus their logs. This is the modified Beer-Lambert quantity and the ratio form is what cancels the per-channel LED drive current and transimpedance gain.
- *Generic bank*: waveform shape and spectral statistics — dispersion, skewness, kurtosis, dominant frequency, band power, pulse rate, inter-beat variability.
- *QC bank*: finite ratio, duration, clipping fraction, peak count, SNR. Excluded from all predictor matrices because recording length and contact quality are acquisition artefacts, not blood properties.

**Stage E — Evaluation scaffolding.** One Hb-quantile-stratified, subject-level, five-fold split is generated once, saved, and reused by every model, every feature bank, every subset and every baseline, so that nothing is ever compared across different splits. Imputation, zero-variance screening, scaling and hyperparameter search all happen strictly inside each outer training partition.

**Stage F — RQ1, Optical Information Gain.** Fit demographic-only, optical-only and combined models on identical folds. OIG is the improvement in out-of-fold R-squared of combined over demographic-only. The null is generated by permuting the *optical block only* across subjects, which breaks the optics-to-Hb link while preserving the demographics-to-Hb link, then refitting. The permutation p-value is the fraction of null OIG values at least as large as the observed one.

**Stage G — RQ2, representation comparison.** Physics bank versus generic bank versus their union, then a small 1D-CNN on raw four-channel windows, normalised with training-fold global statistics rather than per-window so that the DC offset survives into the network. Subject-level prediction is the mean over that subject's windows.

**Stage H — Exhaustive ablation.** All fifteen non-empty wavelength subsets, full retraining for every subset, fold and seed, never inference-time masking. Pooled at subject level across seeds, bootstrapped at subject level for confidence intervals, compared to the full model with paired Wilcoxon and Holm correction, and adjudicated against a **pre-registered non-inferiority margin of 0.2 g/dL** rather than by reading a non-significant p-value as equivalence.

**Stage I — RQ3, the attribution audit.** SHAP is computed from the **same tuned estimator and the same seed pool** as the ablation, not a separately fitted default-hyperparameter model. Reference importance comes from leave-one-feature-out with full retraining. Agreement is measured at the feature level, where n is large enough for a rank correlation to have power, rather than across four wavelengths where the smallest attainable two-sided p-value is 0.083. The whole procedure is then repeated on a label-permuted model to produce the null distribution and the Attribution Credibility Index.

**Stage J — RQ4, the photon-budget test.** Empirical wavelength importance is regressed on measured mean SNR, and the ablation is re-run within SNR tertiles.

**Stage K — Specificity control.** The identical pipeline is applied to fasting glucose, SBP and DBP.

**Stage L — Reporting.** Calibration slope, Bland-Altman limits of agreement, anaemia screening at sex-specific thresholds, seed stability, publication-sized figures written as vector PDF plus a 200 dpi PNG preview, all tables as CSV, and a run manifest.

**Stage M — Substantive integrity gates.** The final checks do not merely confirm that the machinery ran. They assert that the model beat the mean baseline, that calibration slope is in a plausible band, and that optical-only beat demographic-only before any wavelength claim is permitted to stand. A gate suite that cannot fail on a null result gives false assurance.

---

## 6. Models

| Role | Model | Notes |
|---|---|---|
| Primary | LightGBM regressor | Randomised search, 20 candidates, 3-fold inner CV, MAE objective |
| Linear reference | RidgeCV | Alpha grid tuned inside the fold; an untuned alpha understates linear models |
| Trivial reference | Training-fold mean | The floor any claim must clear |
| Representation arm | 1D-CNN (PyTorch, CUDA) | 4-channel raw input, GPU arm, subject-level pooling |
| Explainer | TreeSHAP on the tuned estimator | Audited against retrain-and-remove, calibrated against a permuted-label null |

---

## 7. What this notebook does and does not claim

**Supports:** all fifteen wavelength subsets evaluated under one frozen subject-level split; out-of-fold performance throughout; a confound-controlled estimate of optical information with a permutation null; a null-calibrated assessment of whether SHAP rankings are meaningful; regression, agreement, calibration, screening and stability reporting; computational evidence relevant to future reduced-channel system design.

**Does not support:** a validated physical device; a clinically diagnostic model; replacement of the complete blood count; proof that a wavelength is physically unnecessary in all devices and populations; proven non-inferiority beyond the stated margin on this cohort; correlation as proof of agreement.

---

## 8. Hardware target

Acer Nitro 5, Intel Core i5-12500H (12 cores / 16 threads), NVIDIA RTX 3050 Laptop 4 GB, Windows. Tabular models run on CPU with the outer loop parallelised and LightGBM pinned to one thread per worker, because with 252 subjects GPU transfer overhead exceeds any gain. The GPU is used by the 1D-CNN arm, with mixed precision and a memory budget well inside 4 GB. Set `SMOKE_TEST = True` for the first pass.

### Cell 1 - Environment bootstrap
Checks every required package, installs anything missing with `pip`, detects CPU thread count and CUDA availability, and sets thread caps before NumPy is imported so that BLAS does not oversubscribe the outer parallel loop. PyTorch is treated as optional: if it is absent the CNN arm is skipped cleanly rather than crashing the run.

In [1]:
import importlib, os, subprocess, sys
REQUIRED = {"numpy":"numpy","pandas":"pandas","scipy":"scipy","sklearn":"scikit-learn","matplotlib":"matplotlib",
            "lightgbm":"lightgbm","shap":"shap","statsmodels":"statsmodels","joblib":"joblib","openpyxl":"openpyxl"}
missing = [p for m, p in REQUIRED.items() if importlib.util.find_spec(m) is None]
if missing:
    print("Installing:", missing); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
INSTALL_TORCH = True          # set False to skip the GPU arm entirely
if importlib.util.find_spec("torch") is None and INSTALL_TORCH:
    print("Installing PyTorch (CUDA 12.1 wheels, ~2.5 GB) - this runs once.")
    try: subprocess.check_call([sys.executable,"-m","pip","install","-q","torch","--index-url","https://download.pytorch.org/whl/cu121"])
    except Exception as e: print("PyTorch install failed, CNN arm will be skipped:", e)
N_CPU = os.cpu_count() or 8; N_JOBS = max(1, N_CPU - 2)
for v in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS","NUMEXPR_NUM_THREADS"]: os.environ[v] = "1"
HAS_TORCH, GPU_NAME, DEVICE = False, None, "cpu"
try:
    import torch; HAS_TORCH = True
    if torch.cuda.is_available():
        DEVICE, GPU_NAME = "cuda", torch.cuda.get_device_name(0)
        print(f"CUDA: {GPU_NAME} | {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB | torch {torch.__version__}")
    else: print("PyTorch present but no CUDA device; CNN arm will run on CPU (slow).")
except Exception as e: print("PyTorch unavailable, CNN arm disabled:", e)
print(f"CPU threads: {N_CPU} | outer-loop workers: {N_JOBS} | BLAS pinned to 1 thread/worker")

CUDA: NVIDIA GeForce RTX 3050 Laptop GPU | 4.0 GiB | torch 2.11.0+cu128
CPU threads: 16 | outer-loop workers: 14 | BLAS pinned to 1 thread/worker


### Cell 2 - Imports, configuration, paths, figure style
One configuration dictionary drives every downstream setting, so smoke and final runs differ only by the `SMOKE_TEST` flag. Figure defaults are set once for publication output: small physical sizes, 200 dpi PNG previews and vector PDF masters, which keeps an Overleaf project small. The non-inferiority margin `NI_MARGIN` and the permutation counts are declared here rather than buried in the analysis cells, so the pre-registration is visible in one place.

In [2]:
import itertools, json, platform, time, warnings
from datetime import datetime
from pathlib import Path
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as sps, stats as spstats
from scipy.stats import wilcoxon
from joblib import Parallel, delayed
import lightgbm as lgb, shap
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix,
                             mean_absolute_error, mean_squared_error, precision_recall_curve,
                             r2_score, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore", category=UserWarning); warnings.filterwarnings("ignore", category=FutureWarning)

SMOKE_TEST = False                     # <-- flip to False for the final, paper-quality run
DATASET_ROOT_OVERRIDE = None          # e.g. Path(r"D:\Hb_PPG_Dataset")
RESULTS_ROOT_OVERRIDE = None

def detect_root(override, kaggle, win_default):
    if override is not None: return Path(override)
    kp = Path(kaggle)
    if kp.exists():
        cands = [p for p in kp.iterdir() if p.is_dir()]
        if cands: return cands[0]
    return Path(win_default)
DATASET_ROOT = detect_root(DATASET_ROOT_OVERRIDE, "/kaggle/input", r"D:\STUDY MATERIAL\CVPR\Project\Radish_CNN_Project\Hb_PPG_Dataset")
RESULTS_ROOT = Path(RESULTS_ROOT_OVERRIDE) if RESULTS_ROOT_OVERRIDE else (
    Path("/kaggle/working/Hb_PPG_Results") if Path("/kaggle/working").exists()
    else Path(r"D:\STUDY MATERIAL\CVPR\Project\Radish_CNN_Project\Hb_PPG_Results"))
assert DATASET_ROOT.exists(), f"Dataset root not found: {DATASET_ROOT}. Set DATASET_ROOT_OVERRIDE."

CFG = dict(wavelengths=["660","730","850","940"], fs=200, bandpass=(0.5,8.0), filter_order=4,
           snr_band=(0.5,10.0), win_sec=10.0, hop_sec=5.0,
           n_outer_folds=2 if SMOKE_TEST else 5, n_inner_folds=2 if SMOKE_TEST else 3,
           seeds=[0] if SMOKE_TEST else [0,1,2], n_hparam_candidates=3 if SMOKE_TEST else 20,
           n_bootstrap=200 if SMOKE_TEST else 2000, n_perm=20 if SMOKE_TEST else 200,
           n_lofo_top=8 if SMOKE_TEST else 25, n_lofo_rand=8 if SMOKE_TEST else 25,
           max_subjects=40 if SMOKE_TEST else None, hb_bins=3 if SMOKE_TEST else 5,
           cnn_epochs=8 if SMOKE_TEST else 60, cnn_batch=64, cnn_lr=1e-3,
           anemia_thr=dict(M=13.0, F=12.0, U=12.0), random_state=1337)
NI_MARGIN = 0.20                      # pre-registered non-inferiority margin, g/dL
TAG = "smoke" if SMOKE_TEST else "final"
for sub in ["cache","tables","figures","models"]: (RESULTS_ROOT/sub).mkdir(parents=True, exist_ok=True)
FIG_DIR, TBL_DIR = RESULTS_ROOT/"figures", RESULTS_ROOT/"tables"

plt.rcParams.update({"figure.dpi":110, "savefig.dpi":200, "savefig.bbox":"tight", "savefig.pad_inches":0.02,
    "font.size":8, "axes.labelsize":8, "axes.titlesize":8.5, "xtick.labelsize":7, "ytick.labelsize":7,
    "legend.fontsize":7, "axes.grid":True, "grid.alpha":0.25, "grid.linewidth":0.4,
    "axes.spines.top":False, "axes.spines.right":False, "lines.linewidth":1.0, "pdf.fonttype":42})
COL1, COL2 = 3.4, 7.0                 # single- and double-column widths, inches
def savefig(fig, name):
    for ext in ("pdf","png"): fig.savefig(FIG_DIR/f"{name}.{ext}")
    plt.close(fig); return FIG_DIR/f"{name}.pdf"
def parallel_map(fn, args, n_jobs, desc=""):
    # loky occasionally fails to start on Windows notebook kernels; fall back rather than lose the run.
    if n_jobs == 1: return [fn(*a) if isinstance(a, tuple) else fn(a) for a in args]
    try: return Parallel(n_jobs=n_jobs, backend="loky")(delayed(fn)(*a) if isinstance(a, tuple) else delayed(fn)(a) for a in args)
    except Exception as e:
        print(f"Parallel backend failed for {desc} ({type(e).__name__}); running sequentially.")
        return [fn(*a) if isinstance(a, tuple) else fn(a) for a in args]
print(f"Dataset: {DATASET_ROOT}\nResults: {RESULTS_ROOT}\nMode: {'SMOKE TEST' if SMOKE_TEST else 'FINAL RUN'} | "
      f"folds {CFG['n_outer_folds']} | seeds {CFG['seeds']} | perms {CFG['n_perm']} | NI margin {NI_MARGIN} g/dL")

Dataset: D:\STUDY MATERIAL\CVPR\Project\Radish_CNN_Project\Hb_PPG_Dataset
Results: D:\STUDY MATERIAL\CVPR\Project\Radish_CNN_Project\Hb_PPG_Results
Mode: FINAL RUN | folds 5 | seeds [0, 1, 2] | perms 200 | NI margin 0.2 g/dL


### Cell 3 - Dataset discovery and channel-order assertion
Locates the workbook and the numerically named per-subject CSVs. The original pipeline took `numeric_cols[:4]` on faith; that happens to be right for Hb-PPG, but a single upstream file with an index column would relabel all four wavelengths and invalidate every downstream claim without raising anything. `load_signal` therefore resolves columns by header name when the headers carry wavelength information, falls back to positional order only when they do not, and asserts the resulting shape.

In [3]:
def discover_dataset(root: Path):
    wb = sorted(root.rglob("*subject*information*.xls*")) or sorted(root.rglob("*.xlsx")) + sorted(root.rglob("*.xls"))
    assert wb, f"No subject-information workbook under {root}"
    csvs = {int(p.stem): p for p in root.rglob("*.csv") if p.stem.isdigit()}
    assert csvs, f"No numeric subject CSVs under {root}"
    return wb[0], csvs

def resolve_channels(df, wavelengths):
    # Map each wavelength to a column. Prefer header matching; fall back to positional order.
    cols = [str(c) for c in df.columns]; mapping = {}
    for wl in wavelengths:
        hits = [c for c in cols if wl in c.replace(" ", "")]
        if len(hits) == 1: mapping[wl] = hits[0]
    if len(mapping) == len(wavelengths): return mapping, "header"
    num = [c for c in df.columns if pd.to_numeric(df[c], errors="coerce").notna().mean() > 0.9]
    assert len(num) >= len(wavelengths), f"Only {len(num)} numeric columns; expected >= {len(wavelengths)}"
    return {wl: num[i] for i, wl in enumerate(wavelengths)}, "positional"

def load_signal(path: Path, wavelengths):
    df = pd.read_csv(path); mapping, how = resolve_channels(df, wavelengths)
    out = {wl: pd.to_numeric(df[mapping[wl]], errors="coerce").to_numpy(float) for wl in wavelengths}
    n = len(next(iter(out.values())))
    assert n > 500, f"{path.name}: only {n} samples"
    return out, how

workbook_path, csv_by_subject = discover_dataset(DATASET_ROOT)
_probe, _how = load_signal(next(iter(csv_by_subject.values())), CFG["wavelengths"])
print(f"Workbook: {workbook_path.name} | signal files: {len(csv_by_subject)} | channel resolution: {_how}")
print("Probe channel medians (raw ADC counts):", {k: round(float(np.nanmedian(v)), 1) for k, v in _probe.items()})
assert all(np.nanmedian(v) > 0 for v in _probe.values()), (
    "Raw channel medians must be positive ADC counts. A non-positive DC means the file is already "
    "detrended or AC-coupled, and the Beer-Lambert features cannot be formed.")

Workbook: subject information.xlsx | signal files: 252 | channel resolution: header
Probe channel medians (raw ADC counts): {'660': 22126.5, '730': 47738.5, '850': 13099.0, '940': 51154.5}


### Cell 4 - Metadata, unit correction, demographics, auxiliary targets
Loads the workbook by fuzzy column matching, converts Hb from g/L to g/dL when the median indicates it, and enforces a plausible physiological range so the pipeline fails loudly rather than training on the wrong scale. Placeholder strings are treated as missing, never as zero. Fasting glucose and brachial blood pressure are also pulled in, because they are the positive controls for the cross-target specificity analysis in Cell 26.

In [4]:
MISSING = {"/","-","--","","na","n/a","nan","none","null"}
def clean_missing(s): 
    t = s.astype(str).str.strip(); return t.where(~t.str.lower().isin(MISSING), other=np.nan)

def load_metadata(path: Path) -> pd.DataFrame:
    meta = pd.read_excel(path); meta.columns = [str(c).strip() for c in meta.columns]
    def find(*kw, avoid=()):
        for c in meta.columns:
            cl = c.lower()
            if all(k in cl for k in kw) and not any(a in cl for a in avoid): return c
        return None
    col = dict(id=find("subject") or find("id"), hb=find("hb") or find("hemoglobin") or find("haemoglobin"),
               sex=find("sex") or find("gender"), age=find("age"), height=find("height"), weight=find("weight"),
               glucose=find("glucose"), sbp=find("systolic") or find("sbp"), dbp=find("diastolic") or find("dbp"),
               siglen=find("signal","length") or find("length"))
    assert col["id"] and col["hb"], "Could not locate subject-ID and haemoglobin columns"
    out = pd.DataFrame({"subject_id": pd.to_numeric(meta[col["id"]], errors="coerce").astype("Int64"),
                        "hb_raw": pd.to_numeric(clean_missing(meta[col["hb"]]), errors="coerce")})
    if col["sex"]:
        sx = clean_missing(meta[col["sex"]]).str.upper().str[0]
        out["sex"] = sx.where(sx.isin(["M","F"]), other=np.nan)
    else: out["sex"] = np.nan
    for name in ["age","height","weight","glucose","sbp","dbp","siglen"]:
        out[name] = pd.to_numeric(clean_missing(meta[col[name]]), errors="coerce") if col[name] else np.nan
    assert out["subject_id"].notna().all(), "Unparseable subject IDs"
    assert out["subject_id"].duplicated().sum() == 0, "Duplicate subject IDs in metadata"
    return out

meta_df = load_metadata(workbook_path)
median_hb = meta_df["hb_raw"].median()
meta_df["hb_gdl"] = meta_df["hb_raw"]/10.0 if median_hb > 30 else meta_df["hb_raw"]
hb_unit_note = "g/L -> g/dL converted" if median_hb > 30 else "already g/dL"
assert meta_df["hb_gdl"].dropna().between(3, 25).all(), "Hb outside plausible g/dL range after unit correction"
meta_df["bmi"] = meta_df["weight"] / (meta_df["height"]/100.0)**2
DEMO_CANDIDATES = ["sex","age","height","weight","bmi"]
DEMO_COLS = [c for c in DEMO_CANDIDATES
             if meta_df[c].notna().sum() >= max(10, 0.5*len(meta_df)) and meta_df[c].dropna().nunique() >= 2]
AUX_TARGETS = [t for t in ["glucose","sbp","dbp"] if meta_df[t].notna().sum() >= max(30, 0.5*len(meta_df))]
print(f"Hb unit: {hb_unit_note} | median {meta_df['hb_gdl'].median():.2f} g/dL | "
      f"range {meta_df['hb_gdl'].min():.1f}-{meta_df['hb_gdl'].max():.1f}")
print("Demographic predictors:", DEMO_COLS, "| auxiliary targets available:", AUX_TARGETS)
print("Signal-length values present:", sorted(meta_df['siglen'].dropna().unique())[:8] if meta_df['siglen'].notna().any() else "not recorded")

Hb unit: g/L -> g/dL converted | median 13.70 g/dL | range 8.5-17.3
Demographic predictors: ['sex', 'age', 'height', 'weight', 'bmi'] | auxiliary targets available: ['glucose', 'sbp', 'dbp']
Signal-length values present: [np.int64(30), np.int64(35), np.int64(40), np.int64(45), np.int64(50), np.int64(55), np.int64(60)]


### Cell 5 - Match metadata to signal files
Restricts to subjects that have both a usable Hb reference and a signal file. The smoke-test cap is drawn as an outcome-stratified subsample so a quick run still spans the haemoglobin range rather than sampling one tail.

In [5]:
meta_df = meta_df[meta_df["subject_id"].astype(int).isin(csv_by_subject)].dropna(subset=["hb_gdl"]).reset_index(drop=True)
meta_df["subject_id"] = meta_df["subject_id"].astype(int)
if CFG["max_subjects"] and len(meta_df) > CFG["max_subjects"]:
    tmp = meta_df.copy()
    tmp["bin"] = pd.qcut(tmp["hb_gdl"], q=min(CFG["hb_bins"], tmp["hb_gdl"].nunique()), duplicates="drop")
    parts = [g.sample(n=min(max(1, round(CFG["max_subjects"]*len(g)/len(tmp))), len(g)), random_state=CFG["random_state"])
             for _, g in tmp.groupby("bin", observed=True)]
    meta_df = pd.concat(parts, ignore_index=True).drop(columns="bin").drop_duplicates("subject_id").reset_index(drop=True)
assert len(meta_df) >= 10, "Too few matched subjects to proceed"
print(f"Matched subjects: {len(meta_df)} | anaemic by sex-specific threshold: "
      f"{int((meta_df['hb_gdl'] < meta_df['sex'].map(CFG['anemia_thr']).fillna(CFG['anemia_thr']['U'])).sum())}")

Matched subjects: 252 | anaemic by sex-specific threshold: 18


### Cell 6 - Signal conditioning (the correction at the heart of this notebook)
The previous pipeline executed `x = sps.detrend(x)` and then returned that array under the key `"raw"`, so `median(raw)` was approximately zero and `AC/DC` became a division by near-zero noise. Here the raw array is never overwritten: `raw`, `detrended` and `bandpassed` are three separate arrays derived from one interpolation pass. `assert_dc_preserved` is a runtime guard that makes the old bug impossible to reintroduce silently - it fails if the returned DC is not a substantial positive fraction of the raw median.

In [6]:
def preprocess_channel(x, fs, band, order, min_finite=0.5):
    x = np.asarray(x, float).copy(); finite = np.isfinite(x); ratio = float(finite.mean()) if x.size else 0.0
    if ratio < min_finite or finite.sum() < 10: return None, ratio
    if not finite.all():
        idx = np.arange(x.size); x[~finite] = np.interp(idx[~finite], idx[finite], x[finite])
    raw = x.copy()                                    # <-- never reassigned again
    det = sps.detrend(raw, type="linear")
    nyq = fs/2.0; lo, hi = band[0]/nyq, min(band[1], nyq*0.99)/nyq
    b, a = sps.butter(order, [lo, hi], btype="bandpass")
    try: bp = sps.filtfilt(b, a, det)
    except ValueError: bp = det.copy()
    mad = np.median(np.abs(bp - np.median(bp))) or 1e-9
    return {"raw": raw, "detrended": det, "bandpassed": bp, "scaled": (bp - np.median(bp))/(1.4826*mad)}, ratio

def assert_dc_preserved(prep, name=""):
    dc, rawmed = float(np.median(prep["raw"])), float(np.median(prep["raw"]))
    assert dc > 0 and abs(np.median(prep["detrended"])) < 0.25*abs(rawmed) + 1e-9, (
        f"DC guard failed for {name}: raw median {dc:.3g}. The raw array must retain its DC offset "
        f"while the detrended array must not.")
    return True

_p, _r = preprocess_channel(_probe["660"], CFG["fs"], CFG["bandpass"], CFG["filter_order"]); assert_dc_preserved(_p, "660 probe")
print(f"DC guard passed. raw median {np.median(_p['raw']):.1f} | detrended median {np.median(_p['detrended']):.3g} | "
      f"band-passed SD {np.std(_p['bandpassed']):.3f} | finite ratio {_r:.3f}")

DC guard passed. raw median 22126.5 | detrended median 8.97 | band-passed SD 2087.619 | finite ratio 1.000


### Cell 7 - Per-channel signal-to-noise ratio
SNR is computed as the ratio of in-band (0.5-10 Hz) to out-of-band power in decibels, which is the definition the dataset authors used, so our per-channel numbers are directly comparable to the published ones (850 nm cleanest, 940 nm noisiest). These values feed the photon-budget confound test in Cell 25 and are recorded in the quality-control table. They are deliberately excluded from every predictor matrix.

In [7]:
def channel_snr_db(x, fs, band):
    x = np.asarray(x, float); x = x - np.mean(x)
    f, psd = sps.welch(x, fs=fs, nperseg=min(len(x), fs*4))
    inb = (f >= band[0]) & (f <= band[1]); p_in, p_out = psd[inb].sum(), psd[~inb].sum()
    return float(10*np.log10(p_in/p_out)) if p_out > 0 and p_in > 0 else np.nan
print("SNR of probe channels (dB):",
      {wl: round(channel_snr_db(_probe[wl], CFG["fs"], CFG["snr_band"]), 2) for wl in CFG["wavelengths"]})

SNR of probe channels (dB): {'660': 11.97, '730': 13.22, '850': 14.15, '940': 14.48}


### Cell 8 - Three feature banks, kept strictly apart
`PHYS` implements the modified Beer-Lambert quantities: DC from the raw signal, AC from the band-passed signal, their ratio, perfusion index, log-DC, and the six cross-wavelength ratio-of-ratios plus logs. The ratio form is what cancels the per-channel LED drive current and transimpedance gain of the ADPD4100 front end, which is why raw amplitude alone is not comparable across channels. `GEN` holds the generic waveform and spectral statistics that stand in for the old feature set. `QC` holds acquisition artefacts - duration, finite ratio, clipping, peak count, SNR - which are reported but never used as predictors, because recording length is a property of the session and not of the subject's blood.

In [8]:
PHYS_PER_WL = ["dc","log_dc","ac_rms","ac_mad","ac_p2p","ac_dc_mad","ac_dc_rms","perfusion_index"]
def phys_features(prep):
    raw, bp = prep["raw"], prep["bandpassed"]; dc = float(np.median(raw))
    ac_rms = float(np.sqrt(np.mean(bp**2))); ac_mad = float(1.4826*np.median(np.abs(bp - np.median(bp))))
    ac_p2p = float(np.percentile(bp, 99) - np.percentile(bp, 1)); safe = dc if dc > 1e-9 else np.nan
    return dict(dc=dc, log_dc=float(np.log(safe)) if np.isfinite(safe) else np.nan, ac_rms=ac_rms, ac_mad=ac_mad,
                ac_p2p=ac_p2p, ac_dc_mad=ac_mad/safe, ac_dc_rms=ac_rms/safe, perfusion_index=ac_p2p/safe)

GEN_PER_WL = ["filt_std","iqr","skew","kurtosis","crest_factor","dom_freq_hz","band_power","spec_entropy",
              "hr_bpm","ibi_cv","rise_ratio","aug_index"]
def gen_features(prep, fs):
    bp, sc = prep["bandpassed"], prep["scaled"]; sd = float(np.std(bp))
    q75, q25 = np.percentile(bp, [75, 25])
    f, psd = sps.welch(bp, fs=fs, nperseg=min(len(bp), fs*4)); m = (f >= 0.5) & (f <= 8.0)
    if m.any() and psd[m].sum() > 0:
        p = psd[m]/psd[m].sum(); dom, bpow = float(f[m][np.argmax(psd[m])]), float(psd[m].sum())
        ent = float(-(p*np.log(p + 1e-12)).sum()/np.log(len(p)))
    else: dom, bpow, ent = 0.0, 0.0, np.nan
    hb_band = (f >= 0.75) & (f <= 3.0)
    hr = float(f[hb_band][np.argmax(psd[hb_band])]*60.0) if hb_band.any() and psd[hb_band].sum() > 0 else dom*60.0
    pk, _ = sps.find_peaks(sc, distance=int(fs*0.35), prominence=0.3)
    if len(pk) >= 3:
        ibi = np.diff(pk)/fs; ibi_cv = float(np.std(ibi)/np.mean(ibi)) if np.mean(ibi) > 0 else np.nan
        tr, _ = sps.find_peaks(-sc, distance=int(fs*0.35))
        rise = [ (pk[pk > t][0]-t)/fs for t in tr if (pk > t).any() ][:len(pk)]
        rise_ratio = float(np.median(rise)/np.median(ibi)) if rise and np.median(ibi) > 0 else np.nan
        aug = float(np.median(sc[pk])/ (np.percentile(sc, 99) or 1.0))
    else: ibi_cv, rise_ratio, aug = np.nan, np.nan, np.nan
    return dict(filt_std=sd, iqr=float(q75-q25), skew=float(spstats.skew(bp)) if sd > 0 else 0.0,
                kurtosis=float(spstats.kurtosis(bp)) if sd > 0 else 0.0,
                crest_factor=float(np.max(np.abs(bp))/sd) if sd > 0 else np.nan, dom_freq_hz=dom, band_power=bpow,
                spec_entropy=ent, hr_bpm=hr, ibi_cv=ibi_cv, rise_ratio=rise_ratio, aug_index=aug)

def qc_features(prep, finite_ratio, fs, snr_band):
    raw = prep["raw"]; hi, lo = np.quantile(raw, 0.999), np.quantile(raw, 0.001)
    pk, _ = sps.find_peaks(prep["scaled"], distance=int(fs*0.35), prominence=0.3)
    return dict(finite_ratio=finite_ratio, duration_s=len(raw)/fs, peak_count=len(pk),
                clipping_fraction=float(np.mean((raw >= hi) | (raw <= lo))), snr_db=channel_snr_db(raw, fs, snr_band))

CROSS_BASES = ["ac_dc_mad","ac_dc_rms","perfusion_index","dc"]
def build_row(sid, path, wls, cfg):
    try: chans, _ = load_signal(path, wls)
    except Exception as e: return None, f"load_failed: {e}"
    row = {"subject_id": sid}; per = {}
    for wl in wls:
        prep, fr = preprocess_channel(chans[wl], cfg["fs"], cfg["bandpass"], cfg["filter_order"])
        if prep is None: per[wl] = None; row[f"qc{wl}_finite_ratio"] = fr; continue
        p, g, q = phys_features(prep), gen_features(prep, cfg["fs"]), qc_features(prep, fr, cfg["fs"], cfg["snr_band"])
        per[wl] = p
        for k, v in p.items(): row[f"phys{wl}_{k}"] = v
        for k, v in g.items(): row[f"gen{wl}_{k}"] = v
        for k, v in q.items(): row[f"qc{wl}_{k}"] = v
    for a, b in itertools.combinations(wls, 2):
        if per.get(a) is None or per.get(b) is None: continue
        for base in CROSS_BASES:
            va, vb = per[a][base], per[b][base]
            r = va/vb if (vb not in (0, None) and np.isfinite(vb) and vb != 0) else np.nan
            row[f"phys_{a}_{b}_R_{base}"] = r
            row[f"phys_{a}_{b}_logR_{base}"] = float(np.log(r)) if (np.isfinite(r) and r > 0) else np.nan
    return row, None
print(f"Feature banks defined | physics/wl {len(PHYS_PER_WL)} | generic/wl {len(GEN_PER_WL)} | "
      f"cross bases {len(CROSS_BASES)} x 6 pairs x 2 forms = {len(CROSS_BASES)*6*2}")

Feature banks defined | physics/wl 8 | generic/wl 12 | cross bases 4 x 6 pairs x 2 forms = 48


### Cell 9 - Extract all features, with a schema-versioned cache
Extraction is deterministic and cached. The original notebook's cache was not invalidated by code changes, which meant a fixed bug could silently re-run on corrupted features. The cache filename here embeds a `SCHEMA` string: change any feature definition, bump `SCHEMA`, and the old file is ignored automatically rather than relying on the analyst to remember to delete it.

In [9]:
SCHEMA = "v2-dcfix-3bank"
cache_path = RESULTS_ROOT/"cache"/f"features_{SCHEMA}_{TAG}.csv"
def extract_all(meta, csvs, wls, cfg, path: Path, n_jobs=1):
    if path.exists():
        c = pd.read_csv(path)
        if set(meta["subject_id"]).issubset(set(c["subject_id"])):
            print(f"Cache hit: {path.name}"); return c[c["subject_id"].isin(meta["subject_id"])].reset_index(drop=True)
    res = parallel_map(build_row, [(s, csvs[s], wls, cfg) for s in meta["subject_id"]], n_jobs, "feature extraction")
    rows = [r for r, e in res if r is not None]; fails = [(s, e) for (r, e), s in zip(res, meta["subject_id"]) if r is None]
    if fails: print(f"Extraction failed for {len(fails)} subjects:", fails[:5])
    df = pd.DataFrame(rows); df.to_csv(path, index=False); return df

t0 = time.time(); feat_df = extract_all(meta_df, csv_by_subject, CFG["wavelengths"], CFG, cache_path, n_jobs=min(N_JOBS, 8))
PHYS_COLS = sorted([c for c in feat_df.columns if c.startswith("phys")])
GEN_COLS  = sorted([c for c in feat_df.columns if c.startswith("gen")])
QC_COLS   = sorted([c for c in feat_df.columns if c.startswith("qc")])
print(f"Extraction {time.time()-t0:.1f}s | table {feat_df.shape} | physics {len(PHYS_COLS)} | generic {len(GEN_COLS)} | QC {len(QC_COLS)}")
dc_cols = [f"phys{w}_dc" for w in CFG["wavelengths"] if f"phys{w}_dc" in feat_df.columns]
print("DC sanity - median raw ADC level per channel:", {w: round(float(feat_df[f"phys{w}_dc"].median()), 1)
      for w in CFG["wavelengths"] if f"phys{w}_dc" in feat_df.columns})
assert all(feat_df[c].median() > 0 for c in dc_cols), "DC features are not positive: the Beer-Lambert bank is invalid."

Extraction 4.4s | table (252, 149) | physics 80 | generic 48 | QC 20
DC sanity - median raw ADC level per channel: {'660': 22944.2, '730': 19380.8, '850': 23375.0, '940': 44508.0}


### Cell 10 - Assemble the modelling table
Merges metadata, features and targets into one subject-level table and encodes sex numerically for modelling while keeping the raw label for the sex-specific anaemia thresholds. A per-subject mean SNR column is added for the photon-budget analysis. Nothing here enters a predictor matrix by accident: predictor sets are named explicitly in the next cell.

In [10]:
data = meta_df.merge(feat_df, on="subject_id", how="inner").copy()
data["sex_female"] = data["sex"].map({"F":1.0, "M":0.0})
snr_cols = [f"qc{wl}_snr_db" for wl in CFG["wavelengths"] if f"qc{wl}_snr_db" in data.columns]
if snr_cols: data["mean_snr_db"] = data[snr_cols].mean(axis=1)
DEMO_MODEL_COLS = [("sex_female" if c == "sex" else c) for c in DEMO_COLS]
assert data["subject_id"].is_unique and len(data) >= 10
print(f"Modelling table {data.shape} | demographic predictors {DEMO_MODEL_COLS}")
print(data[["hb_gdl"] + [c for c in ["age","bmi","mean_snr_db"] if c in data]].describe().round(2).to_string())

Modelling table (252, 162) | demographic predictors ['sex_female', 'age', 'height', 'weight', 'bmi']
       hb_gdl     age     bmi  mean_snr_db
count  252.00  252.00  175.00       252.00
mean    13.91   47.23   22.71         7.61
std      1.47   20.76    3.12         4.09
min      8.50   21.00   16.10        -2.88
25%     13.00   25.00   20.26         4.76
50%     13.70   46.00   22.72         7.46
75%     15.10   65.00   24.22        10.06
max     17.30   90.00   32.43        19.17


### Cell 11 - Predictor-set definitions and the fifteen wavelength subsets
Wavelength matching is anchored to exact `phys{wl}_`, `gen{wl}_` and `phys_{a}_{b}_` prefixes, so a demographic column such as `weight` can never be mistaken for a wavelength feature. A cross-wavelength feature is admitted to a subset only when both of its participating channels are present, which is what makes the subset comparison honest. All fifteen non-empty subsets are enumerated exhaustively.

In [11]:
WLS = CFG["wavelengths"]
def wl_cols(wl, bank): return [c for c in bank if c.startswith(f"phys{wl}_") or c.startswith(f"gen{wl}_")]
def cross_cols(subset, bank):
    out = []
    for c in bank:
        if not c.startswith("phys_"): continue
        p = c.split("_")
        if len(p) >= 3 and p[1] in subset and p[2] in subset: out.append(c)
    return out
def cols_for_subset(subset, bank): 
    return sorted(set(sum([wl_cols(w, bank) for w in subset], []) + cross_cols(subset, bank)))

OPTICAL_ALL = sorted(PHYS_COLS + GEN_COLS)
BANKS = {"physics": PHYS_COLS, "generic": GEN_COLS, "physics+generic": OPTICAL_ALL}
ALL_SUBSETS = [s for r in range(1, len(WLS)+1) for s in itertools.combinations(WLS, r)]
FULL_NAME = "+".join(WLS)
assert len(ALL_SUBSETS) == 15
assert not (set(OPTICAL_ALL) & set(DEMO_MODEL_COLS)) and not (set(OPTICAL_ALL) & set(QC_COLS)), \
    "Predictor sets must not overlap: optical, demographic and QC blocks are kept disjoint by design."
for name, bank in BANKS.items(): print(f"{name:16s} {len(bank):4d} features | full-subset resolves to {len(cols_for_subset(tuple(WLS), bank))}")
print(f"{len(ALL_SUBSETS)} wavelength subsets | demographic block {len(DEMO_MODEL_COLS)} | QC block {len(QC_COLS)} (excluded from predictors)")

physics            80 features | full-subset resolves to 80
generic            48 features | full-subset resolves to 48
physics+generic   128 features | full-subset resolves to 128
15 wavelength subsets | demographic block 5 | QC block 20 (excluded from predictors)


### Cell 12 - Frozen subject-level outer folds
One Hb-quantile-stratified subject-level split is generated once, written to disk and reused by every model, feature bank, subset, baseline and permutation in this notebook. Freezing the split is what makes the fifteen subset comparisons and the three representation arms commensurable; regenerating folds per analysis would confound subset differences with split differences.

In [12]:
fold_path = TBL_DIR/f"fold_manifest_{TAG}.csv"
def build_folds(df, cfg, path: Path):
    if path.exists():
        fm = pd.read_csv(path)
        if set(fm["subject_id"]) == set(df["subject_id"]): return fm
    bins = pd.qcut(df["hb_gdl"], q=min(cfg["hb_bins"], df["hb_gdl"].nunique()), duplicates="drop", labels=False)
    skf = StratifiedKFold(cfg["n_outer_folds"], shuffle=True, random_state=cfg["random_state"])
    fo = np.empty(len(df), int)
    for k, (_, te) in enumerate(skf.split(df, bins)): fo[te] = k
    fm = pd.DataFrame({"subject_id": df["subject_id"].values, "outer_fold": fo}); fm.to_csv(path, index=False); return fm
fold_manifest = build_folds(data, CFG, fold_path)
data = data.merge(fold_manifest, on="subject_id", how="left"); assert data["outer_fold"].notna().all()
print(data.groupby("outer_fold")["hb_gdl"].agg(["count","mean","std"]).round(2).to_string())

            count   mean   std
outer_fold                    
0              51  13.80  1.72
1              51  13.94  1.34
2              50  13.87  1.61
3              50  13.97  1.27
4              50  13.96  1.43


### Cell 13 - Model factories, fold-safe CV runner, and metrics
Every transformation that can leak is inside the pipeline: median imputation with `keep_empty_features=True` so column identity is preserved for SHAP, zero-variance screening fitted on the training fold only, and for Ridge a standardiser plus a tuned alpha grid. `surviving_cols` recovers the feature names that survive the variance filter, which is what keeps the SHAP name mapping correct - the original notebook's `zip` would have silently misaligned names against values. `run_cv` is the single entry point used by every downstream analysis, so no analysis can accidentally use a different split or a different leakage policy.

In [13]:
GBM_GRID = {"model__n_estimators":[100,200,300,500,800], "model__learning_rate":[0.01,0.02,0.05,0.08,0.1],
            "model__num_leaves":[7,15,31,63], "model__max_depth":[-1,3,4,6,8], "model__min_child_samples":[3,5,10,15,20],
            "model__colsample_bytree":[0.6,0.7,0.8,0.9,1.0], "model__subsample":[0.7,0.85,1.0],
            "model__reg_alpha":[0.0,0.01,0.1,0.5,1.0], "model__reg_lambda":[0.0,0.01,0.1,0.5,1.0]}
def size_aware_grid(n):
    g = {k: list(v) for k, v in GBM_GRID.items()}; cap = max(2, n//8)
    g["model__min_child_samples"] = sorted({min(v, cap) for v in g["model__min_child_samples"]}); return g
def make_gbm(seed, **kw):
    return Pipeline([("imp", SimpleImputer(strategy="median", keep_empty_features=True)), ("var", VarianceThreshold(0.0)),
                     ("model", lgb.LGBMRegressor(random_state=seed, n_jobs=1, verbosity=-1, **kw))])
def make_ridge(seed=0):
    return Pipeline([("imp", SimpleImputer(strategy="median", keep_empty_features=True)), ("var", VarianceThreshold(0.0)),
                     ("sc", StandardScaler()), ("model", RidgeCV(alphas=np.logspace(-3, 4, 30)))])
def surviving_cols(pipe, cols): return list(np.asarray(cols)[pipe.named_steps["var"].get_support()])

def fit_estimator(X, y, seed, cfg, tune=True, params=None):
    pipe = make_gbm(seed, **(params or {}))
    if not tune: return pipe.fit(X, y)
    nb = min(cfg["n_inner_folds"]+1, max(2, len(np.unique(y))))
    bins = pd.qcut(pd.Series(y), q=nb, duplicates="drop", labels=False).to_numpy()
    k = min(cfg["n_inner_folds"], int(pd.Series(bins).value_counts().min()))
    if k < 2: return pipe.fit(X, y)
    cv = list(StratifiedKFold(k, shuffle=True, random_state=seed).split(X, bins))
    s = RandomizedSearchCV(pipe, size_aware_grid(len(y)), n_iter=cfg["n_hparam_candidates"], cv=cv,
                           scoring="neg_mean_absolute_error", random_state=seed, n_jobs=1, refit=True)
    s.fit(X, y); return s.best_estimator_

def run_cv(df, cols, target, cfg, tag, seeds=None, tune=True, estimator="gbm", params=None, n_jobs=1, y_override=None):
    # Out-of-fold predictions on the frozen split. y_override supports permutation nulls.
    seeds = seeds if seeds is not None else cfg["seeds"]
    y_all = pd.Series(y_override, index=df.index) if y_override is not None else df[target]
    def one(fold, seed):
        tr, te = df[df.outer_fold != fold], df[df.outer_fold == fold]
        ytr, yte = y_all.loc[tr.index], y_all.loc[te.index]
        mtr, mte = ytr.notna().to_numpy(), yte.notna().to_numpy()
        if mtr.sum() < 10 or mte.sum() < 2: return None
        Xtr, Xte = tr.loc[mtr, cols], te.loc[mte, cols]
        est = make_ridge(seed).fit(Xtr, ytr[mtr].values) if estimator == "ridge" else \
              fit_estimator(Xtr, ytr[mtr].values, seed, cfg, tune, params)
        return pd.DataFrame(dict(subject_id=te.loc[mte, "subject_id"].values, model=tag, target=target,
                                 outer_fold=fold, seed=seed, y_true=yte[mte].values, y_pred=est.predict(Xte)))
    jobs = [(f, s) for f in range(cfg["n_outer_folds"]) for s in (seeds if estimator != "ridge" else seeds[:1])]
    out = parallel_map(one, jobs, n_jobs, f"run_cv:{tag}")
    out = [o for o in out if o is not None]
    assert out, f"run_cv produced no predictions for {tag}"
    return pd.concat(out, ignore_index=True)

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float); m = np.isfinite(y) & np.isfinite(p); y, p = y[m], p[m]
    if len(y) < 2: return dict(n=len(y), mae=np.nan, rmse=np.nan, r2=np.nan, pearson=np.nan, spearman=np.nan, bias=np.nan)
    return dict(n=len(y), mae=float(mean_absolute_error(y, p)), rmse=float(np.sqrt(mean_squared_error(y, p))),
                r2=float(r2_score(y, p)), pearson=float(spstats.pearsonr(y, p)[0]), spearman=float(spstats.spearmanr(y, p)[0]),
                bias=float(np.mean(p - y)))
def safe_spearman(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float); m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3 or np.unique(x[m]).size < 2 or np.unique(y[m]).size < 2: return np.nan, np.nan
    r, p = spstats.spearmanr(x[m], y[m]); return float(r), float(p)
def pool_subject(oof):
    return oof.groupby(["subject_id","model","target"], as_index=False).agg(y_true=("y_true","mean"), y_pred=("y_pred","mean"))
def boot_ci(y, p, n_boot, rs, stat="mae"):
    y, p = np.asarray(y, float), np.asarray(p, float); rng = np.random.default_rng(rs); n = len(y); vals = np.empty(n_boot)
    for b in range(n_boot):
        i = rng.integers(0, n, n)
        vals[b] = np.mean(np.abs(y[i]-p[i])) if stat == "mae" else r2_score(y[i], p[i])
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
print("Pipelines, CV runner and metrics ready. All preprocessing is fitted inside training folds only.")

Pipelines, CV runner and metrics ready. All preprocessing is fitted inside training folds only.


---
## Contribution 1 - Optical Information Gain

### Cell 14 - Three-arm run: demographic-only, optical-only, combined
The single most important number in this study. The three arms share the frozen split, the same estimator, the same tuning budget and the same leakage policy, so the only thing that varies is the predictor block. Reporting the demographic-only arm is what almost no paper in this field does, and it is the reference against which any optical claim must be judged. A fourth arm regresses Hb on demographics and then models the residual from optics, which is the same question asked a second way.

In [14]:
OIG_ARMS = {"demographic_only": DEMO_MODEL_COLS, "optical_only": OPTICAL_ALL,
            "optical_plus_demographic": sorted(set(OPTICAL_ALL + DEMO_MODEL_COLS))}
oof_arms = pd.concat([run_cv(data, cols, "hb_gdl", CFG, tag, n_jobs=N_JOBS) for tag, cols in OIG_ARMS.items()],
                     ignore_index=True)
pooled_arms = pool_subject(oof_arms)
arm_rows = []
for tag, g in pooled_arms.groupby("model"):
    m = metrics(g.y_true, g.y_pred); _, lo, hi = boot_ci(g.y_true, g.y_pred, CFG["n_bootstrap"], CFG["random_state"])
    _, r2lo, r2hi = boot_ci(g.y_true, g.y_pred, CFG["n_bootstrap"], CFG["random_state"], stat="r2")
    arm_rows.append(dict(model=tag, **m, mae_ci_lo=lo, mae_ci_hi=hi, r2_ci_lo=r2lo, r2_ci_hi=r2hi))
mean_oof = pd.concat([pd.DataFrame(dict(subject_id=data.loc[data.outer_fold == f, "subject_id"].values, model="mean_baseline",
            target="hb_gdl", y_true=data.loc[data.outer_fold == f, "hb_gdl"].values,
            y_pred=data.loc[data.outer_fold != f, "hb_gdl"].mean())) for f in range(CFG["n_outer_folds"])], ignore_index=True)
ridge_oof = pool_subject(run_cv(data, OPTICAL_ALL, "hb_gdl", CFG, "ridge_optical", estimator="ridge", n_jobs=N_JOBS))
for g, tag in [(mean_oof, "mean_baseline"), (ridge_oof, "ridge_optical")]:
    m = metrics(g.y_true, g.y_pred); _, lo, hi = boot_ci(g.y_true, g.y_pred, CFG["n_bootstrap"], CFG["random_state"])
    _, r2lo, r2hi = boot_ci(g.y_true, g.y_pred, CFG["n_bootstrap"], CFG["random_state"], stat="r2")
    arm_rows.append(dict(model=tag, **m, mae_ci_lo=lo, mae_ci_hi=hi, r2_ci_lo=r2lo, r2_ci_hi=r2hi))
arm_summary = pd.DataFrame(arm_rows).sort_values("mae").reset_index(drop=True)
arm_summary.to_csv(TBL_DIR/f"t1_information_arms_{TAG}.csv", index=False)
get_r2 = lambda t: float(arm_summary.loc[arm_summary.model == t, "r2"].iloc[0])
OIG_R2 = get_r2("optical_plus_demographic") - get_r2("demographic_only")
OIG_MAE = float(arm_summary.loc[arm_summary.model == "demographic_only", "mae"].iloc[0]) - \
          float(arm_summary.loc[arm_summary.model == "optical_plus_demographic", "mae"].iloc[0])
print(arm_summary[["model","n","mae","mae_ci_lo","mae_ci_hi","r2","r2_ci_lo","r2_ci_hi","pearson"]].round(4).to_string(index=False))
print(f"\nObserved Optical Information Gain: dR2 = {OIG_R2:+.4f} | dMAE = {OIG_MAE:+.4f} g/dL")

                   model   n    mae  mae_ci_lo  mae_ci_hi      r2  r2_ci_lo  r2_ci_hi  pearson
        demographic_only 252 0.8141     0.7256     0.9071  0.4458    0.3638    0.5242   0.6681
optical_plus_demographic 252 0.8615     0.7704     0.9565  0.3823    0.2932    0.4646   0.6183
           ridge_optical 252 1.1624     1.0574     1.2678  0.0028   -0.0571    0.0514   0.1263
           mean_baseline 252 1.1738     1.0645     1.2829 -0.0011   -0.0213    0.0008  -0.0435
            optical_only 252 1.1756     1.0616     1.2897 -0.0489   -0.1463    0.0344   0.1352

Observed Optical Information Gain: dR2 = -0.0636 | dMAE = -0.0475 g/dL


### Cell 15 - The permutation null for Optical Information Gain
A raw gain of a few thousandths of R-squared means nothing without a null. The correct null here is **not** to permute the target, which would destroy the demographic signal too. It permutes the **optical block only** across subjects, breaking the optics-to-Hb association while leaving the demographics-to-Hb association intact, then refits the combined arm end to end. The p-value is the fraction of permuted gains at least as large as the observed one, with the standard `(count + 1)/(n + 1)` correction so it can never be reported as exactly zero.

In [15]:
def one_permutation(seed_i):
    rng = np.random.default_rng(CFG["random_state"] + 1000 + seed_i)
    perm = data.copy(); idx = rng.permutation(len(perm))
    perm[OPTICAL_ALL] = perm[OPTICAL_ALL].to_numpy()[idx]                      # shuffle optics, keep demo+target aligned
    comb = pool_subject(run_cv(perm, OIG_ARMS["optical_plus_demographic"], "hb_gdl", CFG, "perm",
                               seeds=[CFG["seeds"][0]], n_jobs=1))
    return r2_score(comb.y_true, comb.y_pred) - get_r2("demographic_only")
t0 = time.time()
null_oig = np.array(parallel_map(one_permutation, list(range(CFG["n_perm"])), N_JOBS, "OIG permutation null"))
p_oig = (np.sum(null_oig >= OIG_R2) + 1) / (len(null_oig) + 1)
z_oig = (OIG_R2 - null_oig.mean()) / (null_oig.std(ddof=1) + 1e-12)
oig_tbl = pd.DataFrame([dict(observed_dR2=OIG_R2, observed_dMAE=OIG_MAE, null_mean=float(null_oig.mean()),
    null_sd=float(null_oig.std(ddof=1)), null_p97_5=float(np.percentile(null_oig, 97.5)), z=float(z_oig),
    p_perm=float(p_oig), n_perm=len(null_oig), significant=bool(p_oig < 0.05))])
oig_tbl.to_csv(TBL_DIR/f"t2_oig_permutation_{TAG}.csv", index=False)
np.save(TBL_DIR/f"oig_null_{TAG}.npy", null_oig)
print(f"{len(null_oig)} permutations in {time.time()-t0:.0f}s")
print(oig_tbl.round(4).to_string(index=False))
print("\nReading:", "optics add information beyond demographics" if p_oig < 0.05 else
      "the optical block adds nothing distinguishable from shuffled optics on this cohort")

200 permutations in 1521s
 observed_dR2  observed_dMAE  null_mean  null_sd  null_p97_5       z  p_perm  n_perm  significant
      -0.0636        -0.0475    -0.0223   0.0225      0.0166 -1.8324  0.9453     200        False

Reading: the optical block adds nothing distinguishable from shuffled optics on this cohort


---
## Contribution 2 - Representation comparison

### Cell 16 - Beer-Lambert physics bank versus generic waveform statistics
Three predictor banks on identical folds. This converts the preprocessing correction into a controlled experiment rather than a footnote: if the physics bank beats the generic bank, the AC/DC quantity is carrying information that waveform statistics miss, which is a result. If both are flat, that is a stronger negative than either alone would be, because it rules out "they used the wrong features" as an explanation.

In [16]:
bank_rows, bank_oof = [], []
for name, bank in BANKS.items():
    cols = cols_for_subset(tuple(WLS), bank)
    o = run_cv(data, cols, "hb_gdl", CFG, f"bank::{name}", n_jobs=N_JOBS); bank_oof.append(o)
    g = pool_subject(o); m = metrics(g.y_true, g.y_pred)
    _, lo, hi = boot_ci(g.y_true, g.y_pred, CFG["n_bootstrap"], CFG["random_state"])
    bank_rows.append(dict(bank=name, n_features=len(cols), **m, mae_ci_lo=lo, mae_ci_hi=hi))
bank_oof = pd.concat(bank_oof, ignore_index=True); bank_summary = pd.DataFrame(bank_rows)
bank_summary.to_csv(TBL_DIR/f"t3_feature_banks_{TAG}.csv", index=False)
pb = pool_subject(bank_oof)
wide = pb.pivot_table(index="subject_id", columns="model", values=["y_true","y_pred"])
err = {b: np.abs(wide[("y_true", f"bank::{b}")] - wide[("y_pred", f"bank::{b}")]) for b in BANKS}
stat, p_pg = wilcoxon(err["physics"] - err["generic"])
bank_summary["vs_generic_wilcoxon_p"] = [p_pg if b == "physics" else np.nan for b in bank_summary.bank]
print(bank_summary[["bank","n_features","mae","mae_ci_lo","mae_ci_hi","r2","pearson"]].round(4).to_string(index=False))
print(f"\nPhysics vs generic, paired Wilcoxon on per-subject absolute error: p = {p_pg:.4f} | "
      f"mean paired difference {np.mean(err['physics']-err['generic']):+.4f} g/dL")

           bank  n_features    mae  mae_ci_lo  mae_ci_hi      r2  pearson
        physics          80 1.2282     1.1128     1.3395 -0.0909   0.0543
        generic          48 1.1592     1.0524     1.2712 -0.0297   0.1267
physics+generic         128 1.1756     1.0616     1.2897 -0.0489   0.1352

Physics vs generic, paired Wilcoxon on per-subject absolute error: p = 0.1155 | mean paired difference +0.0690 g/dL


### Cell 17 - Learned representation arm: 1D-CNN on raw four-channel PPG (GPU)
The third representation. Signals are cut into 10 s windows with 5 s hop, and normalised with **training-fold global per-channel statistics rather than per-window statistics**, which is a deliberate design choice: per-window standardisation would delete the DC offset and reintroduce exactly the failure this notebook corrects. Subject-level prediction is the mean over that subject's windows, and windows never cross the subject-level fold boundary. Mixed precision keeps the footprint far inside 4 GB; the model is small because 252 subjects cannot support a large one, and the arm exists to test whether a learned representation recovers information the handcrafted banks miss, not to win a leaderboard.

In [17]:
CNN_DONE = False
if HAS_TORCH:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    def subject_windows(sid, path, cfg):
        ch, _ = load_signal(path, cfg["wavelengths"]); mats = []
        for wl in cfg["wavelengths"]:
            p, _ = preprocess_channel(ch[wl], cfg["fs"], cfg["bandpass"], cfg["filter_order"])
            mats.append(p["raw"] if p is not None else np.full(1, np.nan))
        n = min(len(m) for m in mats); X = np.stack([m[:n] for m in mats]).astype(np.float32)
        w, h = int(cfg["win_sec"]*cfg["fs"]), int(cfg["hop_sec"]*cfg["fs"])
        return np.stack([X[:, s:s+w] for s in range(0, n-w+1, h)]) if n >= w else np.empty((0, 4, w), np.float32)
    t0 = time.time()
    win_cache = {int(s): subject_windows(int(s), csv_by_subject[int(s)], CFG) for s in data["subject_id"]}
    win_cache = {k: v for k, v in win_cache.items() if len(v) > 0}
    print(f"Windowed {len(win_cache)} subjects in {time.time()-t0:.0f}s | "
          f"{sum(len(v) for v in win_cache.values())} windows | shape {next(iter(win_cache.values())).shape[1:]}")
    class PPGNet(nn.Module):
        def __init__(s, c=4):
            super().__init__()
            blk = lambda i, o, k, st: nn.Sequential(nn.Conv1d(i, o, k, st, k//2), nn.BatchNorm1d(o), nn.GELU(), nn.MaxPool1d(2))
            s.f = nn.Sequential(blk(c,32,7,2), blk(32,64,5,2), blk(64,96,3,1), nn.AdaptiveAvgPool1d(1), nn.Flatten())
            s.h = nn.Sequential(nn.Dropout(0.3), nn.Linear(96, 32), nn.GELU(), nn.Linear(32, 1))
        def forward(s, x): return s.h(s.f(x)).squeeze(-1)
    def cnn_fold(fold, seed):
        torch.manual_seed(seed); np.random.seed(seed)
        tr = data[(data.outer_fold != fold) & data.subject_id.isin(win_cache)]
        te = data[(data.outer_fold == fold) & data.subject_id.isin(win_cache)]
        Xtr = np.concatenate([win_cache[s] for s in tr.subject_id]); ytr = np.concatenate(
            [np.full(len(win_cache[s]), h, np.float32) for s, h in zip(tr.subject_id, tr.hb_gdl)])
        mu = Xtr.mean(axis=(0,2), keepdims=True); sd = Xtr.std(axis=(0,2), keepdims=True) + 1e-6   # train-fold global stats
        dl = DataLoader(TensorDataset(torch.from_numpy((Xtr-mu)/sd), torch.from_numpy(ytr)),
                        batch_size=CFG["cnn_batch"], shuffle=True, drop_last=len(Xtr) > CFG["cnn_batch"])
        net = PPGNet().to(DEVICE); opt = torch.optim.AdamW(net.parameters(), lr=CFG["cnn_lr"], weight_decay=1e-2)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG["cnn_epochs"]); lossf = nn.L1Loss()
        amp = DEVICE == "cuda"
        try: scaler, actx = torch.amp.GradScaler("cuda", enabled=amp), (lambda: torch.amp.autocast("cuda", enabled=amp))
        except (AttributeError, TypeError): scaler, actx = torch.cuda.amp.GradScaler(enabled=amp), (lambda: torch.cuda.amp.autocast(enabled=amp))
        net.train()
        for _ in range(CFG["cnn_epochs"]):
            for xb, yb in dl:
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True); opt.zero_grad(set_to_none=True)
                with actx(): loss = lossf(net(xb), yb)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            sch.step()
        net.eval(); rows = []
        with torch.no_grad():
            for s, h in zip(te.subject_id, te.hb_gdl):
                xb = torch.from_numpy((win_cache[s]-mu)/sd).to(DEVICE)
                with actx(): pr = net(xb).float().cpu().numpy()
                rows.append(dict(subject_id=s, model="cnn_raw_ppg", target="hb_gdl", outer_fold=fold, seed=seed,
                                 y_true=float(h), y_pred=float(np.mean(pr))))
        if DEVICE == "cuda": torch.cuda.empty_cache()
        return pd.DataFrame(rows)
    t0 = time.time()
    cnn_oof = pd.concat([cnn_fold(f, s) for f in range(CFG["n_outer_folds"]) for s in CFG["seeds"]], ignore_index=True)
    cnn_pooled = pool_subject(cnn_oof); cnn_m = metrics(cnn_pooled.y_true, cnn_pooled.y_pred)
    _, clo, chi = boot_ci(cnn_pooled.y_true, cnn_pooled.y_pred, CFG["n_bootstrap"], CFG["random_state"])
    cnn_row = dict(bank="cnn_raw_ppg(learned)", n_features=np.nan, **cnn_m, mae_ci_lo=clo, mae_ci_hi=chi)
    bank_summary = pd.concat([bank_summary, pd.DataFrame([cnn_row])], ignore_index=True)
    bank_summary.to_csv(TBL_DIR/f"t3_feature_banks_{TAG}.csv", index=False); CNN_DONE = True
    print(f"CNN arm done in {time.time()-t0:.0f}s on {DEVICE.upper()} | " +
          " | ".join(f"{k} {v:.4f}" for k, v in cnn_m.items() if k in ("mae","r2","pearson")))
else:
    print("PyTorch not available - CNN arm skipped. Install with:\n"
          "  pip install torch --index-url https://download.pytorch.org/whl/cu121")

Windowed 252 subjects in 3s | 2584 windows | shape (4, 2000)
CNN arm done in 233s on CUDA | mae 1.3556 | r2 -0.3588 | pearson 0.0124


---
## Contribution 5 (part) - Exhaustive ablation with a pre-registered margin

### Cell 18 - Exhaustive fifteen-subset ablation
Every combination of subset, outer fold and seed gets a complete retrain with hyperparameter search inside the outer training partition - never inference-time masking, which would leak the full-model fit into the reduced condition. The outer loop is parallelised across `N_JOBS` workers with LightGBM pinned to one thread inside each, because nesting two parallel layers causes thread oversubscription and makes this slower on a 12-core laptop, not faster. The chosen bank is whichever of physics or physics+generic performed better in Cell 16, recorded explicitly so the choice is auditable.

In [18]:
ABLATION_BANK = "physics+generic" if bank_summary.set_index("bank").loc["physics+generic","mae"] <= \
                                     bank_summary.set_index("bank").loc["physics","mae"] else "physics"
BANK = BANKS[ABLATION_BANK]; print(f"Ablation bank: {ABLATION_BANK} ({len(BANK)} features)")
def ablate_one(subset, fold, seed):
    cols = cols_for_subset(subset, BANK)
    tr, te = data[data.outer_fold != fold], data[data.outer_fold == fold]
    est = fit_estimator(tr[cols], tr["hb_gdl"].values, seed, CFG, tune=True)
    bp = est.named_steps["model"].get_params()
    return pd.DataFrame(dict(subject_id=te["subject_id"].values, subset="+".join(subset), n_wavelengths=len(subset),
        outer_fold=fold, seed=seed, y_true=te["hb_gdl"].values, y_pred=est.predict(te[cols]),
        best_params=json.dumps({k: bp[k] for k in ("n_estimators","learning_rate","num_leaves","max_depth")})))
t0 = time.time()
tasks = [(s, f, sd) for s in ALL_SUBSETS for f in range(CFG["n_outer_folds"]) for sd in CFG["seeds"]]
oof_ablation = pd.concat(parallel_map(ablate_one, tasks, N_JOBS, "exhaustive ablation"), ignore_index=True)
print(f"Ablation: {len(tasks)} fits in {time.time()-t0:.0f}s | {len(oof_ablation)} predictions")
assert oof_ablation["subset"].nunique() == 15 and oof_ablation[["y_true","y_pred"]].notna().all().all()

Ablation bank: physics+generic (128 features)
Ablation: 225 fits in 172s | 11340 predictions


### Cell 19 - Subject-level pooling and bootstrap confidence intervals
Seeds are pooled at the subject level first, then the bootstrap resamples **subjects**, not predictions, because the effective independent sample size is the number of people and not the number of rows. Resampling rows would produce intervals that are far too narrow and would make noise-level subset differences look real.

In [19]:
pooled_abl = (oof_ablation.groupby(["subject_id","subset","n_wavelengths"], as_index=False)
              .agg(y_true=("y_true","mean"), y_pred=("y_pred","mean")))
rows = []
for name, g in pooled_abl.groupby("subset"):
    m = metrics(g.y_true, g.y_pred); bm, lo, hi = boot_ci(g.y_true, g.y_pred, CFG["n_bootstrap"], CFG["random_state"])
    rows.append(dict(subset=name, n_wavelengths=int(g.n_wavelengths.iloc[0]), **m,
                     mae_boot_mean=bm, mae_ci_lo=lo, mae_ci_hi=hi))
subset_summary = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
subset_summary.to_csv(TBL_DIR/f"t4_subset_summary_{TAG}.csv", index=False)
full_mae = float(subset_summary.loc[subset_summary.subset == FULL_NAME, "mae"].iloc[0])
print(subset_summary[["subset","n_wavelengths","mae","mae_ci_lo","mae_ci_hi","r2"]].round(4).to_string(index=False))
print(f"\nSpread across all 15 subsets: {subset_summary.mae.max()-subset_summary.mae.min():.4f} g/dL | "
      f"typical CI width: {(subset_summary.mae_ci_hi-subset_summary.mae_ci_lo).mean():.4f} g/dL")

         subset  n_wavelengths    mae  mae_ci_lo  mae_ci_hi      r2
        730+850              2 1.1557     1.0423     1.2642 -0.0153
        730+940              2 1.1597     1.0507     1.2682 -0.0153
        660+730              2 1.1604     1.0493     1.2670  0.0031
            730              1 1.1663     1.0591     1.2731 -0.0169
            660              1 1.1740     1.0657     1.2826 -0.0273
    660+730+940              3 1.1755     1.0621     1.2843 -0.0424
660+730+850+940              4 1.1756     1.0616     1.2897 -0.0489
    660+730+850              3 1.1757     1.0655     1.2850 -0.0294
        660+940              2 1.1840     1.0717     1.2967 -0.0664
            940              1 1.1867     1.0752     1.3021 -0.0653
    730+850+940              3 1.1895     1.0774     1.2995 -0.0662
            850              1 1.1987     1.0896     1.3103 -0.0765
        850+940              2 1.2082     1.0951     1.3245 -0.1066
        660+850              2 1.2110     1.1023

### Cell 20 - Paired comparison and the pre-registered non-inferiority test
Two separate questions, deliberately not conflated. The Wilcoxon signed-rank test with Holm correction asks whether a reduced subset differs from the full model. The non-inferiority test asks the question that actually matters for channel reduction: a subset is declared non-inferior only when the **upper bound of the 95 % bootstrap CI on the paired MAE difference falls below the pre-registered margin of 0.2 g/dL**. A non-significant p-value is never treated as evidence of equivalence, which is the error the original notebook's selection rule was one step away from making.

In [20]:
full_idx = pooled_abl[pooled_abl.subset == FULL_NAME].set_index("subject_id")
rng = np.random.default_rng(CFG["random_state"]); rows = []
for name, g in pooled_abl.groupby("subset"):
    if name == FULL_NAME: continue
    g = g.set_index("subject_id"); common = g.index.intersection(full_idx.index)
    er = np.abs(g.loc[common,"y_true"] - g.loc[common,"y_pred"]).to_numpy()
    ef = np.abs(full_idx.loc[common,"y_true"] - full_idx.loc[common,"y_pred"]).to_numpy(); d = er - ef
    try: stat, p = wilcoxon(d)
    except ValueError: stat, p = np.nan, 1.0
    bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(CFG["n_bootstrap"])])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    rows.append(dict(subset=name, n_wavelengths=int(g.n_wavelengths.iloc[0]), mean_paired_diff=float(d.mean()),
        diff_ci_lo=float(lo), diff_ci_hi=float(hi), wilcoxon_stat=stat, p_raw=float(p),
        non_inferior=bool(hi < NI_MARGIN)))
paired = pd.DataFrame(rows)
paired["p_holm"] = multipletests(paired.p_raw, method="holm")[1]
paired["significant_holm"] = paired.p_holm < 0.05
paired = paired.merge(subset_summary[["subset","mae"]], on="subset").sort_values(["n_wavelengths","mae"])
paired.to_csv(TBL_DIR/f"t5_paired_noninferiority_{TAG}.csv", index=False)
ni = paired[paired.non_inferior].sort_values(["n_wavelengths","mae"])
SELECTED = ni.iloc[0]["subset"] if len(ni) else subset_summary.iloc[0]["subset"]
SELECT_REASON = (f"smallest subset non-inferior at margin {NI_MARGIN} g/dL" if len(ni)
                 else "no subset met the non-inferiority margin; reporting best-performing configuration only")
print(paired[["subset","n_wavelengths","mae","mean_paired_diff","diff_ci_lo","diff_ci_hi","p_holm","non_inferior"]].round(4).to_string(index=False))
print(f"\nSelected: {SELECTED} | {SELECT_REASON}")

     subset  n_wavelengths    mae  mean_paired_diff  diff_ci_lo  diff_ci_hi  p_holm  non_inferior
        730              1 1.1663           -0.0094     -0.0575      0.0386  1.0000          True
        660              1 1.1740           -0.0016     -0.0505      0.0458  1.0000          True
        940              1 1.1867            0.0111     -0.0407      0.0651  1.0000          True
        850              1 1.1987            0.0231     -0.0281      0.0732  1.0000          True
    730+850              2 1.1557           -0.0199     -0.0649      0.0212  1.0000          True
    730+940              2 1.1597           -0.0159     -0.0586      0.0268  1.0000          True
    660+730              2 1.1604           -0.0153     -0.0519      0.0204  1.0000          True
    660+940              2 1.1840            0.0083     -0.0329      0.0489  1.0000          True
    850+940              2 1.2082            0.0325     -0.0164      0.0811  0.7101          True
    660+850         

---
## Contribution 3 - The Null-Calibrated Attribution Audit

### Cell 21 - Feature-level SHAP from the ablation estimator
Two corrections to the original design are applied here. First, SHAP is computed from the **same tuned estimator and the same seed pool** used to produce the ablation errors, not from a separately fitted default-hyperparameter model - comparing the attributions of one model against the held-out errors of another was guaranteed to disagree. Second, feature names are recovered through `surviving_cols` after the variance filter, and a hard shape assertion makes silent misalignment impossible. Attribution is aggregated at the feature level, where n is in the hundreds, because at the wavelength level n = 4 and the smallest attainable two-sided Spearman p-value is 2/24 = 0.083.

In [21]:
ABL_COLS = cols_for_subset(tuple(WLS), BANK)
def shap_importance(df, cols, cfg, y_override=None, seeds=None, tune=True):
    seeds = seeds or cfg["seeds"]; acc, keep = {}, None
    y_all = pd.Series(y_override, index=df.index) if y_override is not None else df["hb_gdl"]
    for fold in range(cfg["n_outer_folds"]):
        tr, te = df[df.outer_fold != fold], df[df.outer_fold == fold]
        for seed in seeds:
            est = fit_estimator(tr[cols], y_all.loc[tr.index].values, seed, cfg, tune=tune)
            names = surviving_cols(est, cols)
            Xte = est.named_steps["var"].transform(est.named_steps["imp"].transform(te[cols]))
            sv = shap.TreeExplainer(est.named_steps["model"]).shap_values(Xte)
            assert sv.shape[1] == len(names), f"SHAP/name misalignment: {sv.shape[1]} vs {len(names)}"
            for nme, val in zip(names, np.abs(sv).mean(axis=0)): acc[nme] = acc.get(nme, 0.0) + float(val)
    n = cfg["n_outer_folds"]*len(seeds)
    return pd.Series({k: v/n for k, v in acc.items()}).sort_values(ascending=False)
t0 = time.time(); shap_imp = shap_importance(data, ABL_COLS, CFG)
shap_imp.rename("mean_abs_shap").to_frame().to_csv(TBL_DIR/f"t6_shap_feature_importance_{TAG}.csv")
print(f"SHAP over {len(shap_imp)} features in {time.time()-t0:.0f}s\nTop 10:")
print(shap_imp.head(10).round(5).to_string())

SHAP over 128 features in 182s
Top 10:
gen730_rise_ratio                 0.12294
gen730_ibi_cv                     0.11052
gen850_rise_ratio                 0.07652
gen660_skew                       0.07400
gen940_skew                       0.06575
phys_660_850_R_perfusion_index    0.05510
gen660_rise_ratio                 0.05429
gen940_aug_index                  0.05153
phys660_dc                        0.05019
phys730_ac_dc_mad                 0.04347


### Cell 22 - Reference importance by leave-one-feature-out with full retraining
The gold standard a feature attribution should agree with is the change in out-of-fold error when the feature is removed and the model is refitted from scratch. Retraining across every feature would be prohibitive, so the audit set is the top `n_lofo_top` SHAP-ranked features plus `n_lofo_rand` randomly drawn others, which spans the importance range while keeping the comparison honest - restricting the audit to top-ranked features alone would bias the rank correlation upward. Hyperparameters are fixed to the full-feature tuned configuration so that differences reflect feature removal, not a re-drawn hyperparameter search.

In [22]:
tr0 = data[data.outer_fold != 0]
BASE_EST = fit_estimator(tr0[ABL_COLS], tr0["hb_gdl"].values, CFG["seeds"][0], CFG, tune=True)
BASE_PARAMS = {k.replace("model__",""): v for k, v in BASE_EST.get_params().items()
               if k.startswith("model__") and k.split("__")[-1] in
               ("n_estimators","learning_rate","num_leaves","max_depth","min_child_samples",
                "colsample_bytree","subsample","reg_alpha","reg_lambda")}
BASE_PARAMS["min_child_samples"] = min(BASE_PARAMS.get("min_child_samples", 20), max(2, len(tr0)//8))
def lofo_importance(df, cols, cfg, audit, y_override=None, n_jobs=1):
    base = pool_subject(run_cv(df, cols, "hb_gdl", cfg, "lofo_base", tune=False, params=BASE_PARAMS,
                               n_jobs=n_jobs, y_override=y_override))
    base_mae = mean_absolute_error(base.y_true, base.y_pred)
    def one(f):
        red = [c for c in cols if c != f]
        g = pool_subject(run_cv(df, red, "hb_gdl", cfg, "lofo", tune=False, params=BASE_PARAMS, n_jobs=1, y_override=y_override))
        return f, mean_absolute_error(g.y_true, g.y_pred) - base_mae
    out = parallel_map(one, list(audit), n_jobs, "LOFO")
    return pd.Series(dict(out)).sort_values(ascending=False), base_mae
rs = np.random.default_rng(CFG["random_state"])
top = list(shap_imp.head(CFG["n_lofo_top"]).index)
rest = [c for c in ABL_COLS if c not in top]
AUDIT_SET = top + list(rs.choice(rest, size=min(CFG["n_lofo_rand"], len(rest)), replace=False))
t0 = time.time(); lofo_imp, lofo_base_mae = lofo_importance(data, ABL_COLS, CFG, AUDIT_SET, n_jobs=N_JOBS)
audit = pd.DataFrame({"feature": AUDIT_SET, "shap": [shap_imp.get(f, 0.0) for f in AUDIT_SET],
                      "lofo_mae_increase": [lofo_imp.get(f, np.nan) for f in AUDIT_SET]}).dropna()
audit["shap_rank"] = audit.shap.rank(ascending=False); audit["lofo_rank"] = audit.lofo_mae_increase.rank(ascending=False)
RHO_OBS, P_OBS = safe_spearman(audit.shap_rank, audit.lofo_rank)
assert np.isfinite(RHO_OBS), ("LOFO importances are constant: removing any single feature changed nothing. "
    "This happens when the model produced no splits. Increase n_lofo_top/n_lofo_rand or subject count.")
audit.to_csv(TBL_DIR/f"t7_attribution_audit_{TAG}.csv", index=False)
print(f"LOFO over {len(AUDIT_SET)} features in {time.time()-t0:.0f}s | base MAE {lofo_base_mae:.4f}")
print(f"SHAP vs LOFO agreement: Spearman rho = {RHO_OBS:+.3f} (p = {P_OBS:.4f}, n = {len(audit)})")

LOFO over 50 features in 13s | base MAE 1.2399
SHAP vs LOFO agreement: Spearman rho = -0.291 (p = 0.0401, n = 50)


### Cell 23 - The null calibration and the Attribution Credibility Index
This is the contribution. The entire SHAP-plus-LOFO procedure is re-run on models trained against a **permuted haemoglobin target**, so the models provably contain no information about Hb. Each repetition yields a null rank agreement. Two things are then read off. The distribution of null agreements shows whether a meaningless model still produces a coherent-looking attribution story. And the **Attribution Credibility Index** - the observed agreement expressed as a z-score against that null - states how far the real model's explanation sits from what a model that learned nothing would produce. An ACI near zero means the SHAP ranking carries no more evidence of mechanism than chance, which is a warning the SHAP output itself cannot give you.

In [23]:
def null_audit_once(i):
    rng = np.random.default_rng(CFG["random_state"] + 5000 + i)
    y_perm = data["hb_gdl"].to_numpy()[rng.permutation(len(data))]
    si = shap_importance(data, ABL_COLS, CFG, y_override=y_perm, seeds=[CFG["seeds"][0]])
    a_top = list(si.head(CFG["n_lofo_top"]).index)
    a_rest = [c for c in ABL_COLS if c not in a_top]
    a_set = a_top + list(rng.choice(a_rest, size=min(CFG["n_lofo_rand"], len(a_rest)), replace=False))
    li, _ = lofo_importance(data, ABL_COLS, CFG, a_set, y_override=y_perm, n_jobs=1)
    d = pd.DataFrame({"s": [si.get(f, 0.0) for f in a_set], "l": [li.get(f, np.nan) for f in a_set]}).dropna()
    if len(d) < 5: return np.nan, float(si.max()) if len(si) else np.nan
    return safe_spearman(d.s.rank(ascending=False), d.l.rank(ascending=False))[0], float(si.max())
N_NULL = max(5, CFG["n_perm"]//4)
t0 = time.time()
null_out = parallel_map(null_audit_once, list(range(N_NULL)), N_JOBS, "null attribution audit")
null_rho = np.array([r for r, _ in null_out if np.isfinite(r)])
null_top_shap = np.array([s for _, s in null_out if np.isfinite(s)])
DEGENERATE = len(null_rho) < 3
if DEGENERATE:
    ACI, p_aci = np.nan, np.nan
    print(f"Null audit DEGENERATE: only {len(null_rho)}/{N_NULL} repetitions produced a defined rank correlation.\n"
          "Under permuted labels the model produced no splits, so leave-one-feature-out changed nothing.\n"
          "That is itself informative, but the ACI is undefined. Raise n_lofo_top/n_lofo_rand, or re-run with "
          "SMOKE_TEST = False so each training fold has enough subjects to split.")
else:
    ACI = (RHO_OBS - null_rho.mean()) / (null_rho.std(ddof=1) + 1e-12)
    p_aci = (np.sum(null_rho >= RHO_OBS) + 1) / (len(null_rho) + 1)
ncaa = pd.DataFrame([dict(rho_observed=float(RHO_OBS),
    rho_null_mean=float(null_rho.mean()) if len(null_rho) else np.nan,
    rho_null_sd=float(null_rho.std(ddof=1)) if len(null_rho) > 1 else np.nan,
    rho_null_p97_5=float(np.percentile(null_rho, 97.5)) if len(null_rho) else np.nan,
    ACI_z=float(ACI) if np.isfinite(ACI) else np.nan, p_perm=float(p_aci) if np.isfinite(p_aci) else np.nan,
    n_null=len(null_rho), n_null_attempted=N_NULL, degenerate=bool(DEGENERATE), n_audit_features=len(audit),
    null_model_max_mean_abs_shap=float(null_top_shap.mean()) if len(null_top_shap) else np.nan,
    real_model_max_mean_abs_shap=float(shap_imp.max()),
    attribution_credible=bool(np.isfinite(p_aci) and p_aci < 0.05))])
ncaa.to_csv(TBL_DIR/f"t8_null_calibrated_attribution_{TAG}.csv", index=False)
np.save(TBL_DIR/f"ncaa_null_rho_{TAG}.npy", null_rho)
print(ncaa.round(4).to_string(index=False))
print(f"\n{N_NULL} null audits in {time.time()-t0:.0f}s")
print("Reading:", "undefined - see the degeneracy note above" if DEGENERATE else
      ("the SHAP ranking agrees with retrain-and-remove beyond chance" if p_aci < 0.05 else
       "the SHAP ranking is statistically indistinguishable from that of a model trained on shuffled labels"))
if len(null_top_shap):
    print(f"Note for the paper: the label-permuted models, which contain no Hb information by construction, still "
          f"produced a maximum mean |SHAP| of {null_top_shap.mean():.4f} against {shap_imp.max():.4f} for the real model.")

 rho_observed  rho_null_mean  rho_null_sd  rho_null_p97_5   ACI_z  p_perm  n_null  n_null_attempted  degenerate  n_audit_features  null_model_max_mean_abs_shap  real_model_max_mean_abs_shap  attribution_credible
      -0.2913         0.0445       0.1806          0.3392 -1.8599  0.9608      50                50       False                50                         0.089                        0.1229                 False

50 null audits in 956s
Reading: the SHAP ranking is statistically indistinguishable from that of a model trained on shuffled labels
Note for the paper: the label-permuted models, which contain no Hb information by construction, still produced a maximum mean |SHAP| of 0.0890 against 0.1229 for the real model.


### Cell 24 - Wavelength-level attribution versus empirical channel removal
The wavelength-level view is retained for continuity with the prior literature, but reported with the honest caveat attached. Cross-wavelength attribution is split evenly between the two participating channels, which is a convention and not a measurement - and it does not match the removal operator, since dropping a channel also removes all three of its pair features. With four items the rank test has no power by construction, so the Spearman value is reported descriptively and the feature-level audit in Cell 23 carries the inferential weight.

In [24]:
def wl_level_shap(imp_series, wls):
    w = {x: 0.0 for x in wls}
    for col, val in imp_series.items():
        if col.startswith("phys_"):
            p = col.split("_")
            if len(p) >= 3 and p[1] in w and p[2] in w: w[p[1]] += val/2; w[p[2]] += val/2
        else:
            for x in wls:
                if col.startswith(f"phys{x}_") or col.startswith(f"gen{x}_"): w[x] += val; break
    return pd.Series(w).sort_values(ascending=False)
wl_shap = wl_level_shap(shap_imp, WLS)
rem = pd.DataFrame([dict(wavelength=w, mae_without=float(subset_summary.loc[
        subset_summary.subset == "+".join([x for x in WLS if x != w]), "mae"].iloc[0])) for w in WLS])
rem["mae_increase"] = rem.mae_without - full_mae
rem["empirical_rank"] = rem.mae_increase.rank(ascending=False).astype(int)
rem["shap_value"] = rem.wavelength.map(wl_shap); rem["shap_rank"] = rem.shap_value.rank(ascending=False).astype(int)
rho_wl, p_wl = safe_spearman(rem.empirical_rank, rem.shap_rank)
rem.to_csv(TBL_DIR/f"t9_wavelength_attribution_{TAG}.csv", index=False)
print(rem.round(5).to_string(index=False))
print(f"\nWavelength-level Spearman rho = {rho_wl:+.2f}, p = {p_wl:.3f} (n = 4; minimum attainable two-sided p is 0.083, "
      f"so this test cannot reach alpha = 0.05 and is reported descriptively only)")

wavelength  mae_without  mae_increase  empirical_rank  shap_value  shap_rank
       660      1.18954       0.01391               2     0.55827          1
       730      1.21241       0.03678               1     0.54859          2
       850      1.17547      -0.00016               4     0.36244          3
       940      1.17570       0.00007               3     0.34631          4

Wavelength-level Spearman rho = +0.60, p = 0.400 (n = 4; minimum attainable two-sided p is 0.083, so this test cannot reach alpha = 0.05 and is reported descriptively only)


---
## Contribution 4 - The photon-budget confound test

### Cell 25 - Does wavelength importance track SNR rather than haemoglobin optics?
A competing explanation for any apparent wavelength ranking is that it reflects how many usable photons that channel delivered, not how informative that wavelength is about haemoglobin. Empirical importance is therefore regressed on the measured mean SNR per channel, and the full ablation is re-run within SNR tertiles. If the same subset ordering holds in the cleanest and noisiest thirds of the cohort, the ranking is robust; if it flips, any channel-reduction recommendation derived from it is an artefact of the photon budget.

In [25]:
snr_by_wl = pd.Series({w: float(data[f"qc{w}_snr_db"].mean()) for w in WLS if f"qc{w}_snr_db" in data}) if snr_cols else pd.Series(dtype=float)
photon = rem.merge(snr_by_wl.rename("mean_snr_db"), left_on="wavelength", right_index=True, how="left")
if photon.mean_snr_db.notna().sum() >= 3:
    r_si, p_si = safe_spearman(photon.mean_snr_db, photon.shap_value)
    r_se, p_se = safe_spearman(photon.mean_snr_db, photon.mae_increase)
else: r_si = p_si = r_se = p_se = np.nan
single_mae = subset_summary[subset_summary.n_wavelengths == 1].set_index("subset")["mae"]
r_sm, p_sm = safe_spearman(snr_by_wl.reindex(single_mae.index), single_mae) if len(single_mae) == len(snr_by_wl) else (np.nan, np.nan)
tert_rows = []
if "mean_snr_db" in data:
    data["snr_tertile"] = pd.qcut(data["mean_snr_db"], 3, labels=["low","mid","high"], duplicates="drop")
    tsub = pooled_abl.merge(data[["subject_id","snr_tertile"]], on="subject_id", how="left")
    for (t, s), g in tsub.groupby(["snr_tertile","subset"], observed=True):
        if len(g) >= 5: tert_rows.append(dict(tertile=str(t), subset=s, n=len(g), mae=float(mean_absolute_error(g.y_true, g.y_pred))))
tertile_df = pd.DataFrame(tert_rows)
if len(tertile_df):
    piv = tertile_df.pivot(index="subset", columns="tertile", values="mae")
    order_stab = piv.corr(method="spearman")
    print("Subset-ordering stability across SNR tertiles (Spearman between tertile MAE rankings):")
    print(order_stab.round(3).to_string())
photon_tbl = pd.DataFrame([dict(snr_vs_shap_rho=r_si, snr_vs_shap_p=p_si, snr_vs_removal_rho=r_se, snr_vs_removal_p=p_se,
                                snr_vs_single_channel_mae_rho=r_sm, snr_vs_single_channel_mae_p=p_sm)])
photon_tbl.to_csv(TBL_DIR/f"t10_photon_budget_{TAG}.csv", index=False)
if len(tertile_df): tertile_df.to_csv(TBL_DIR/f"t11_snr_tertile_ablation_{TAG}.csv", index=False)
print("\nMean SNR by wavelength (dB):", snr_by_wl.round(2).to_dict())
print(photon_tbl.round(3).to_string(index=False))

Subset-ordering stability across SNR tertiles (Spearman between tertile MAE rankings):
tertile   high    low    mid
tertile                     
high     1.000  0.604 -0.157
low      0.604  1.000  0.218
mid     -0.157  0.218  1.000

Mean SNR by wavelength (dB): {'660': 7.16, '730': 7.52, '850': 8.16, '940': 7.61}
 snr_vs_shap_rho  snr_vs_shap_p  snr_vs_removal_rho  snr_vs_removal_p  snr_vs_single_channel_mae_rho  snr_vs_single_channel_mae_p
            -0.8            0.2                -0.8               0.2                            0.8                          0.2


---
## Contribution 5 - Cross-target specificity control

### Cell 26 - The same pipeline applied to glucose and blood pressure
The positive control that single-target studies cannot provide. Fasting glucose, systolic and diastolic pressure are recorded for these same subjects and these same recordings. Running the identical features, folds and estimator against all four targets separates two very different conclusions: features that are inert for every physiological target point to an information ceiling in this acquisition, whereas features that recover blood pressure but not haemoglobin show the pipeline is sound and that haemoglobin is specifically hard from reflectance PPG. Each target is also run against demographics alone, so the optical gain is reported per target rather than assumed.

In [26]:
spec_rows = []
for tgt in ["hb_gdl"] + AUX_TARGETS:
    if data[tgt].notna().sum() < 30: continue
    for arm, cols in [("optical", OPTICAL_ALL), ("demographic", DEMO_MODEL_COLS)]:
        g = pool_subject(run_cv(data, cols, tgt, CFG, f"{tgt}::{arm}", n_jobs=N_JOBS))
        m = metrics(g.y_true, g.y_pred); _, r2lo, r2hi = boot_ci(g.y_true, g.y_pred, CFG["n_bootstrap"], CFG["random_state"], stat="r2")
        sd = float(data[tgt].std())
        spec_rows.append(dict(target=tgt, arm=arm, **m, r2_ci_lo=r2lo, r2_ci_hi=r2hi,
                              target_sd=sd, mae_over_sd=m["mae"]/sd if sd else np.nan))
specificity = pd.DataFrame(spec_rows)
specificity.to_csv(TBL_DIR/f"t12_cross_target_specificity_{TAG}.csv", index=False)
print(specificity[["target","arm","n","mae","mae_over_sd","r2","r2_ci_lo","r2_ci_hi","pearson"]].round(3).to_string(index=False))
opt_wins = specificity[(specificity.arm == "optical") & (specificity.r2_ci_lo > 0)]["target"].tolist()
print("\nTargets where optical-only R2 has a lower CI bound above zero:", opt_wins or "none")

 target         arm   n    mae  mae_over_sd     r2  r2_ci_lo  r2_ci_hi  pearson
 hb_gdl     optical 252  1.176        0.798 -0.049    -0.146     0.034    0.135
 hb_gdl demographic 252  0.814        0.553  0.446     0.364     0.524    0.668
glucose     optical 217  0.773        0.632 -0.104    -0.261    -0.027    0.018
glucose demographic 217  0.678        0.554  0.081    -0.019     0.159    0.294
    sbp     optical 241 15.557        0.726  0.128     0.010     0.225    0.371
    sbp demographic 241 12.731        0.594  0.388     0.278     0.476    0.623
    dbp     optical 241  8.736        0.657  0.283     0.185     0.372    0.537
    dbp demographic 241  7.788        0.585  0.420     0.318     0.506    0.648

Targets where optical-only R2 has a lower CI bound above zero: ['sbp', 'dbp']


### Cell 27 - Calibration, tolerance bands and Bland-Altman agreement
Correlation is never presented as proof of numerical agreement. Calibration slope and intercept are reported explicitly - a slope near zero identifies a model emitting a near-constant prediction, which is the signature the original run showed and which correlation alone would have hidden. Bland-Altman bias and limits of agreement are computed for both the full and the selected subsets.

In [27]:
def agreement(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float); sl, ic, r, _, _ = spstats.linregress(y, p); d = p - y
    return dict(calib_slope=float(sl), calib_intercept=float(ic), **{f"within_{t}": float(np.mean(np.abs(d) <= t)) for t in (0.5,1.0,1.5)},
                ba_bias=float(d.mean()), ba_loa_lo=float(d.mean()-1.96*d.std()), ba_loa_hi=float(d.mean()+1.96*d.std()))
full_g = pooled_abl[pooled_abl.subset == FULL_NAME]; sel_g = pooled_abl[pooled_abl.subset == SELECTED]
agr = pd.DataFrame([dict(config=FULL_NAME, **agreement(full_g.y_true, full_g.y_pred)),
                    dict(config=SELECTED, **agreement(sel_g.y_true, sel_g.y_pred))])
agr.to_csv(TBL_DIR/f"t13_calibration_agreement_{TAG}.csv", index=False)
print(agr.round(4).to_string(index=False))
if abs(agr.loc[0,"calib_slope"]) < 0.10:
    print("\nWARNING: calibration slope near zero. The model is emitting a near-constant prediction; "
          "any correlation or subset ranking reported from it describes noise.")

         config  calib_slope  calib_intercept  within_0.5  within_1.0  within_1.5  ba_bias  ba_loa_lo  ba_loa_hi
660+730+850+940       0.0533           13.189      0.2817      0.5119      0.7103   0.0221    -2.9288     2.9729
            730       0.0514           13.202      0.2698      0.5198      0.7183   0.0097    -2.8961     2.9154



### Cell 28 - Anaemia screening metrics
Sex-specific thresholds where sex is known, with a single 12 g/dL threshold as a labelled sensitivity analysis otherwise. Results are retrospective algorithmic screening performance, not a diagnostic claim. Given the low anaemia prevalence in this cohort these metrics are explicitly underpowered and are reported with the positive-class count attached so a reader can see exactly how many events the sensitivity is based on.

In [28]:
def screening(g, meta):
    m = g.merge(meta[["subject_id","sex"]], on="subject_id", how="left")
    m["thr"] = m.sex.map(CFG["anemia_thr"]).fillna(CFG["anemia_thr"]["U"])
    yt, yp, sc = (m.y_true < m.thr).astype(int), (m.y_pred < m.thr).astype(int), (m.thr - m.y_pred)
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
    try: au, ap = roc_auc_score(yt, sc), average_precision_score(yt, sc)
    except ValueError: au, ap = np.nan, np.nan
    return dict(n_positive=int(yt.sum()), prevalence=float(yt.mean()), sensitivity=tp/(tp+fn) if tp+fn else np.nan,
                specificity=tn/(tn+fp) if tn+fp else np.nan, balanced_accuracy=float(balanced_accuracy_score(yt, yp)),
                auroc=au, auprc=ap, tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp)), m, yt, sc
scr, scr_m, scr_y, scr_s = screening(full_g, meta_df)
screen_tbl = pd.DataFrame([dict(config=FULL_NAME, **scr)])
screen_tbl.to_csv(TBL_DIR/f"t14_screening_{TAG}.csv", index=False)
print(screen_tbl.round(4).to_string(index=False))
print(f"\nUnderpowered by design: {scr['n_positive']} positive cases. Report as exploratory.")

         config  n_positive  prevalence  sensitivity  specificity  balanced_accuracy  auroc  auprc  tn  fp  fn  tp
660+730+850+940          18      0.0714          0.0       0.9915             0.4957 0.5185 0.0802 232   2  18   0

Underpowered by design: 18 positive cases. Report as exploratory.


### Cell 29 - Seed stability
Per-subset MAE across seeds. The point of this table is not reassurance but scale: the seed-to-seed standard deviation is the noise floor against which subset differences must be judged. If the spread between subsets is smaller than the spread between seeds within a subset, no wavelength ranking derived from those differences is defensible, and the notebook says so explicitly.

In [29]:
seed_tbl = (oof_ablation.assign(ae=lambda d: (d.y_true - d.y_pred).abs())
            .groupby(["subset","seed"], as_index=False)["ae"].mean().rename(columns={"ae":"mae"}))
seed_piv = seed_tbl.pivot(index="subset", columns="seed", values="mae")
seed_tbl.to_csv(TBL_DIR/f"t15_seed_stability_{TAG}.csv", index=False)
SEED_SD = float(seed_piv.std(axis=1).mean()) if seed_piv.shape[1] > 1 else np.nan
SUBSET_SPREAD = float(subset_summary.mae.max() - subset_summary.mae.min())
print(seed_piv.round(4).to_string())
print(f"\nMean within-subset seed SD: {SEED_SD:.4f} g/dL | between-subset spread: {SUBSET_SPREAD:.4f} g/dL "
      f"| ratio {SUBSET_SPREAD/SEED_SD:.2f}" if np.isfinite(SEED_SD) and SEED_SD > 0 else "")
if np.isfinite(SEED_SD) and SUBSET_SPREAD < 2*SEED_SD:
    print("WARNING: between-subset spread is within twice the seed noise. Wavelength rankings from these "
          "differences are not interpretable and must not be presented as a channel-reduction recommendation.")

seed                  0       1       2
subset                                 
660              1.1772  1.1821  1.1879
660+730          1.2021  1.1574  1.1693
660+730+850      1.1885  1.1772  1.1815
660+730+850+940  1.2193  1.1585  1.1751
660+730+940      1.2033  1.1847  1.1754
660+850          1.1893  1.2340  1.2393
660+850+940      1.2176  1.2025  1.2353
660+940          1.1986  1.1818  1.1925
730              1.1791  1.1569  1.1824
730+850          1.1762  1.1575  1.1529
730+850+940      1.1977  1.2013  1.1853
730+940          1.1546  1.1678  1.1676
850              1.1941  1.1928  1.2203
850+940          1.2005  1.1926  1.2461
940              1.1771  1.2067  1.2003

Mean within-subset seed SD: 0.0156 g/dL | between-subset spread: 0.0567 g/dL | ratio 3.63


### Cell 30 - Figures 1 to 5: signal, cohort, SNR, information gain, representations
Every figure is written as a vector PDF master for LaTeX plus a 200 dpi PNG preview. Physical sizes are set to single-column (3.4 in) or double-column (7.0 in) widths so nothing needs rescaling in the manuscript, and the PDFs stay small enough that a full figure set will not strain a free Overleaf project. Figure 1 is deliberately built to show the DC offset surviving into the raw trace, which is the visual proof that the correction in Cell 6 took effect.

In [30]:
FIGS = {}
sid0 = int(data.subject_id.iloc[0]); ch0, _ = load_signal(csv_by_subject[sid0], WLS)
fig, axes = plt.subplots(4, 2, figsize=(COL2, 4.6), sharex=True)
for i, wl in enumerate(WLS):
    p, _ = preprocess_channel(ch0[wl], CFG["fs"], CFG["bandpass"], CFG["filter_order"]); t = np.arange(len(p["raw"]))/CFG["fs"]
    s = slice(0, min(len(t), CFG["fs"]*8))
    axes[i,0].plot(t[s], p["raw"][s], color="0.25"); axes[i,0].set_ylabel(f"{wl} nm")
    axes[i,1].plot(t[s], p["bandpassed"][s], color="C0")
    for a in axes[i]: a.tick_params(labelbottom=(i == 3))
axes[0,0].set_title("Raw (DC preserved)"); axes[0,1].set_title("Band-passed 0.5-8 Hz (AC)")
axes[3,0].set_xlabel("time (s)"); axes[3,1].set_xlabel("time (s)")
fig.suptitle(f"Four-wavelength PPG, subject {sid0}", y=0.995, fontsize=9); FIGS["fig1_signal"] = savefig(fig, "fig1_signal")

fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.1))
ax[0].hist(data.hb_gdl, bins=18, color="0.4"); ax[0].set_xlabel("Hb (g/dL)"); ax[0].set_ylabel("subjects")
for s, c in [("M","C0"),("F","C1")]:
    v = data.loc[data.sex == s, "age"].dropna()
    if len(v): ax[1].hist(v, bins=14, alpha=0.6, label=s, color=c)
ax[1].set_xlabel("age (years)"); ax[1].legend(title="sex")
miss = data[[c for c in ["sex","age","height","weight"]+AUX_TARGETS if c in data]].isna().mean()
ax[2].barh(range(len(miss)), miss.values, color="0.5"); ax[2].set_yticks(range(len(miss))); ax[2].set_yticklabels(miss.index)
ax[2].set_xlabel("fraction missing"); FIGS["fig2_cohort"] = savefig(fig, "fig2_cohort")

if snr_cols:
    fig, ax = plt.subplots(figsize=(COL1, 2.2))
    _v, _l = [data[f"qc{w}_snr_db"].dropna() for w in WLS], [f"{w}" for w in WLS]
    try: ax.boxplot(_v, tick_labels=_l, widths=0.6)
    except TypeError: ax.boxplot(_v, labels=_l, widths=0.6)
    ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("SNR (dB)"); ax.axhline(10, ls="--", lw=0.7, color="C3")
    FIGS["fig3_snr"] = savefig(fig, "fig3_snr")

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.4))
a = arm_summary.set_index("model").reindex([m for m in ["mean_baseline","ridge_optical","optical_only","demographic_only","optical_plus_demographic"] if m in arm_summary.model.values])
y = np.arange(len(a))
ax[0].barh(y, a.mae, xerr=[a.mae-a.mae_ci_lo, a.mae_ci_hi-a.mae], color="0.45", height=0.6, error_kw=dict(lw=0.8))
ax[0].set_yticks(y); ax[0].set_yticklabels([m.replace("_","\n") for m in a.index], fontsize=6.5)
ax[0].set_xlabel("out-of-fold MAE (g/dL)"); ax[0].invert_yaxis()
ax[1].hist(null_oig, bins=24, color="0.7", label="permuted optics")
ax[1].axvline(OIG_R2, color="C3", lw=1.4, label=f"observed  z={z_oig:.2f}")
ax[1].set_xlabel(r"Optical Information Gain ($\Delta R^2$)"); ax[1].set_ylabel("permutations")
ax[1].legend(loc="upper left"); ax[1].set_title(f"permutation p = {p_oig:.3f}")
FIGS["fig4_oig"] = savefig(fig, "fig4_oig")

fig, ax = plt.subplots(figsize=(COL1, 2.2))
b = bank_summary.sort_values("mae"); y = np.arange(len(b))
ax.barh(y, b.mae, xerr=[b.mae-b.mae_ci_lo, b.mae_ci_hi-b.mae], color="0.45", height=0.6, error_kw=dict(lw=0.8))
ax.axvline(float(arm_summary.loc[arm_summary.model == "demographic_only","mae"].iloc[0]), color="C3", ls="--", lw=1,
           label="demographics only")
ax.axvline(float(arm_summary.loc[arm_summary.model == "mean_baseline","mae"].iloc[0]), color="0.2", ls=":", lw=1, label="mean baseline")
ax.set_yticks(y); ax.set_yticklabels(b.bank, fontsize=6.5); ax.set_xlabel("MAE (g/dL)"); ax.invert_yaxis(); ax.legend(fontsize=6)
FIGS["fig5_banks"] = savefig(fig, "fig5_banks")
print("Figures 1-5 written:", list(FIGS))

Figures 1-5 written: ['fig1_signal', 'fig2_cohort', 'fig3_snr', 'fig4_oig', 'fig5_banks']


### Cell 31 - Figures 6 to 10: ablation frontier, attribution audit, null calibration, photon budget
Figure 7 is the one to lead with in the explainability section: SHAP importance against retrain-and-remove importance at the feature level, with the observed rank correlation annotated. Figure 8 places that correlation inside the distribution produced by label-permuted models, which is the visual statement of the Attribution Credibility Index.

In [31]:
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.8))
o = subset_summary.sort_values("mae"); y = np.arange(len(o))
ax[0].errorbar(o.mae, y, xerr=[o.mae-o.mae_ci_lo, o.mae_ci_hi-o.mae], fmt="o", ms=2.5, lw=0.7, color="0.3")
ax[0].axvline(full_mae, color="C0", ls="--", lw=0.9, label="all four")
ax[0].axvspan(full_mae, full_mae+NI_MARGIN, color="C2", alpha=0.12, label=f"NI margin {NI_MARGIN}")
ax[0].set_yticks(y); ax[0].set_yticklabels(o.subset, fontsize=5.8); ax[0].set_xlabel("MAE (g/dL)"); ax[0].legend(fontsize=6)
bp = subset_summary.loc[subset_summary.groupby("n_wavelengths").mae.idxmin()].sort_values("n_wavelengths")
ax[1].errorbar(bp.n_wavelengths, bp.mae, yerr=[bp.mae-bp.mae_ci_lo, bp.mae_ci_hi-bp.mae], marker="o", ms=3, lw=0.9, color="0.3")
ax[1].set_xlabel("number of wavelengths"); ax[1].set_ylabel("best MAE (g/dL)"); ax[1].set_xticks([1,2,3,4])
FIGS["fig6_ablation"] = savefig(fig, "fig6_ablation")

fig, ax = plt.subplots(figsize=(COL1, 2.4))
ax.scatter(audit.shap, audit.lofo_mae_increase, s=9, color="0.3")
ax.axhline(0, lw=0.6, color="C3", ls="--"); ax.set_xlabel("mean |SHAP|"); ax.set_ylabel("LOFO MAE increase (g/dL)")
ax.set_title(fr"$\rho$ = {RHO_OBS:+.2f}, p = {P_OBS:.3f}, n = {len(audit)}")
FIGS["fig7_shap_vs_lofo"] = savefig(fig, "fig7_shap_vs_lofo")

if len(null_rho) >= 3:
    fig, ax = plt.subplots(figsize=(COL1, 2.2))
    ax.hist(null_rho, bins=min(18, max(5, len(null_rho)//2)), color="0.7", label="label-permuted models")
    ax.axvline(RHO_OBS, color="C3", lw=1.5, label=f"observed  ACI z={ACI:.2f}")
    ax.set_xlabel(r"SHAP vs LOFO rank agreement ($\rho$)"); ax.set_ylabel("null repetitions")
    ax.legend(fontsize=6, loc="upper left"); ax.set_title(f"permutation p = {p_aci:.3f}")
    FIGS["fig8_ncaa"] = savefig(fig, "fig8_ncaa")
else: print("Figure 8 skipped: the null audit was degenerate (see Cell 23).")

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.3))
w = np.arange(len(rem)); wd = 0.38
ax[0].bar(w-wd/2, rem.shap_rank, wd, label="SHAP rank", color="0.4")
ax[0].bar(w+wd/2, rem.empirical_rank, wd, label="removal rank", color="0.7")
ax[0].set_xticks(w); ax[0].set_xticklabels(rem.wavelength); ax[0].set_ylabel("rank (1 = most important)")
ax[0].set_xlabel("wavelength (nm)"); ax[0].legend(fontsize=6); ax[0].set_title(fr"$\rho$={rho_wl:+.2f} (n=4, no power)")
if photon.mean_snr_db.notna().any():
    ax[1].scatter(photon.mean_snr_db, photon.mae_increase, s=22, color="0.3")
    for _, r in photon.iterrows(): ax[1].annotate(r.wavelength, (r.mean_snr_db, r.mae_increase), fontsize=6, xytext=(3,3), textcoords="offset points")
    ax[1].set_xlabel("mean channel SNR (dB)"); ax[1].set_ylabel("MAE increase on removal")
    ax[1].set_title(fr"$\rho$={r_se:+.2f}, p={p_se:.2f}")
FIGS["fig9_wavelength_photon"] = savefig(fig, "fig9_wavelength_photon")

if len(tertile_df):
    fig, ax = plt.subplots(figsize=(COL1, 2.4))
    piv = tertile_df.pivot(index="subset", columns="tertile", values="mae").reindex(columns=["low","mid","high"])
    im = ax.imshow(piv.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns); ax.set_yticks(range(len(piv)))
    ax.set_yticklabels(piv.index, fontsize=5.5); ax.set_xlabel("SNR tertile"); ax.grid(False)
    fig.colorbar(im, ax=ax, label="MAE (g/dL)"); FIGS["fig10_snr_tertile"] = savefig(fig, "fig10_snr_tertile")
print("Figures 6-10 written.")

Figures 6-10 written.


### Cell 32 - Figures 11 to 14: specificity, agreement, screening, seed stability
Figure 12 pairs the predicted-versus-reference scatter with the Bland-Altman plot on one row, which is the standard agreement display for a method-comparison paper and is the pairing reviewers expect when a correlation is quoted anywhere in the manuscript.

In [32]:
fig, ax = plt.subplots(figsize=(COL1, 2.3))
sp = specificity.pivot(index="target", columns="arm", values="r2")
x = np.arange(len(sp)); wd = 0.38
for i, (arm, c) in enumerate([("optical","0.35"),("demographic","0.7")]):
    if arm in sp: ax.bar(x+(i-0.5)*wd, sp[arm], wd, label=arm, color=c)
ax.axhline(0, lw=0.7, color="C3"); ax.set_xticks(x); ax.set_xticklabels(sp.index, fontsize=6.5)
ax.set_ylabel(r"out-of-fold $R^2$"); ax.legend(fontsize=6); FIGS["fig11_specificity"] = savefig(fig, "fig11_specificity")

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.5))
lims = [data.hb_gdl.min()-1, data.hb_gdl.max()+1]
ax[0].scatter(full_g.y_true, full_g.y_pred, s=8, alpha=0.6, color="0.3")
ax[0].plot(lims, lims, "--", lw=0.8, color="C3"); ax[0].set_xlim(lims); ax[0].set_ylim(lims)
ax[0].set_xlabel("reference Hb (g/dL)"); ax[0].set_ylabel("predicted Hb (g/dL)")
ax[0].set_title(f"slope {agr.loc[0,'calib_slope']:.3f}")
d = full_g.y_pred - full_g.y_true; mn = (full_g.y_pred + full_g.y_true)/2
ax[1].scatter(mn, d, s=8, alpha=0.6, color="0.3")
for v, st, c in [(agr.loc[0,"ba_bias"],"-","0.2"), (agr.loc[0,"ba_loa_lo"],"--","C3"), (agr.loc[0,"ba_loa_hi"],"--","C3")]:
    ax[1].axhline(v, ls=st, lw=0.8, color=c)
ax[1].set_xlabel("mean of methods (g/dL)"); ax[1].set_ylabel("predicted - reference (g/dL)")
ax[1].set_title(f"bias {agr.loc[0,'ba_bias']:+.2f}, LoA [{agr.loc[0,'ba_loa_lo']:.2f}, {agr.loc[0,'ba_loa_hi']:.2f}]")
FIGS["fig12_agreement"] = savefig(fig, "fig12_agreement")

if np.isfinite(scr["auroc"]):
    fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.1))
    fpr, tpr, _ = roc_curve(scr_y, scr_s); pr, rc, _ = precision_recall_curve(scr_y, scr_s)
    ax[0].plot(fpr, tpr, color="0.25"); ax[0].plot([0,1],[0,1],"--",lw=0.7,color="C3")
    ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].set_title(f"AUROC {scr['auroc']:.3f}")
    ax[1].plot(rc, pr, color="0.25"); ax[1].axhline(scr["prevalence"], ls="--", lw=0.7, color="C3")
    ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_title(f"AUPRC {scr['auprc']:.3f}")
    cm = np.array([[scr["tn"], scr["fp"]],[scr["fn"], scr["tp"]]])
    ax[2].imshow(cm, cmap="Greys"); ax[2].grid(False)
    for i in range(2):
        for j in range(2): ax[2].text(j, i, cm[i,j], ha="center", va="center", fontsize=8, color="C3")
    ax[2].set_xticks([0,1]); ax[2].set_xticklabels(["pred -","pred +"]); ax[2].set_yticks([0,1]); ax[2].set_yticklabels(["true -","true +"])
    FIGS["fig13_screening"] = savefig(fig, "fig13_screening")

if seed_piv.shape[1] > 1:
    fig, ax = plt.subplots(figsize=(COL1, 2.6))
    im = ax.imshow(seed_piv.values, aspect="auto", cmap="magma"); ax.grid(False)
    ax.set_xticks(range(seed_piv.shape[1])); ax.set_xticklabels(seed_piv.columns); ax.set_xlabel("seed")
    ax.set_yticks(range(len(seed_piv))); ax.set_yticklabels(seed_piv.index, fontsize=5.5)
    fig.colorbar(im, ax=ax, label="MAE (g/dL)"); FIGS["fig14_seed_stability"] = savefig(fig, "fig14_seed_stability")
sizes = {p.name: p.stat().st_size/1024 for p in sorted(FIG_DIR.glob("*.pdf"))}
print(f"{len(sizes)} vector figures | total {sum(sizes.values())/1024:.2f} MB | largest: "
      f"{max(sizes, key=sizes.get)} at {max(sizes.values()):.0f} KB")
print("All figures also saved as 200 dpi PNG previews in", FIG_DIR)

14 vector figures | total 0.23 MB | largest: fig1_signal.pdf at 39 KB
All figures also saved as 200 dpi PNG previews in D:\STUDY MATERIAL\CVPR\Project\Radish_CNN_Project\Hb_PPG_Results\figures


### Cell 33 - Export tables, headline metrics and the run manifest
Everything a manuscript needs in one place: the headline numbers for the abstract, a feature dictionary mapping every column to its bank and wavelength, the full out-of-fold prediction table for any re-analysis a reviewer requests, and a manifest recording mode, configuration, package versions and hardware for the reproducibility statement.

In [33]:
headline = pd.DataFrame([{
    "n_subjects": len(data), "hb_mean": float(data.hb_gdl.mean()), "hb_sd": float(data.hb_gdl.std()),
    "mean_baseline_mae": float(arm_summary.loc[arm_summary.model=="mean_baseline","mae"].iloc[0]),
    "demographic_only_mae": float(arm_summary.loc[arm_summary.model=="demographic_only","mae"].iloc[0]),
    "demographic_only_r2": get_r2("demographic_only"),
    "optical_only_mae": float(arm_summary.loc[arm_summary.model=="optical_only","mae"].iloc[0]),
    "optical_only_r2": get_r2("optical_only"),
    "combined_mae": float(arm_summary.loc[arm_summary.model=="optical_plus_demographic","mae"].iloc[0]),
    "OIG_dR2": OIG_R2, "OIG_dMAE": OIG_MAE, "OIG_z": float(z_oig), "OIG_p_perm": float(p_oig),
    "best_bank": str(bank_summary.sort_values("mae").bank.iloc[0]),
    "cnn_included": CNN_DONE, "ablation_bank": ABLATION_BANK,
    "full_model_mae": full_mae, "selected_subset": SELECTED, "selection_reason": SELECT_REASON,
    "n_noninferior_subsets": int(paired.non_inferior.sum()), "ni_margin_gdl": NI_MARGIN,
    "shap_lofo_rho": float(RHO_OBS), "shap_lofo_p": float(P_OBS), "ACI_z": float(ACI), "ACI_p": float(p_aci),
    "ncaa_degenerate": bool(DEGENERATE),
    "wavelength_rho": float(rho_wl), "calib_slope": float(agr.loc[0,"calib_slope"]),
    "ba_loa_width": float(agr.loc[0,"ba_loa_hi"]-agr.loc[0,"ba_loa_lo"]),
    "screening_auroc": scr["auroc"], "screening_sensitivity": scr["sensitivity"], "screening_n_positive": scr["n_positive"],
    "seed_sd": SEED_SD, "subset_spread": SUBSET_SPREAD}])
headline.to_csv(TBL_DIR/f"t0_headline_metrics_{TAG}.csv", index=False)
def bank_of(c): return "physics" if c.startswith("phys") else "generic" if c.startswith("gen") else "qc"
def wl_of(c):
    for w in WLS:
        if c.startswith(f"phys{w}_") or c.startswith(f"gen{w}_") or c.startswith(f"qc{w}_"): return w
    p = c.split("_"); return f"{p[1]}|{p[2]}" if c.startswith("phys_") and len(p) >= 3 else "cross/none"
pd.DataFrame([dict(feature=c, bank=bank_of(c), wavelength=wl_of(c), used_as_predictor=c in OPTICAL_ALL,
                   mean_abs_shap=float(shap_imp.get(c, np.nan))) for c in PHYS_COLS+GEN_COLS+QC_COLS]
            ).to_csv(TBL_DIR/f"t16_feature_dictionary_{TAG}.csv", index=False)
oof_ablation.to_csv(TBL_DIR/f"oof_ablation_{TAG}.csv", index=False)
oof_arms.to_csv(TBL_DIR/f"oof_information_arms_{TAG}.csv", index=False)
import sklearn, scipy
manifest = dict(timestamp=datetime.now().isoformat(), mode=TAG, schema=SCHEMA, n_subjects=len(data),
    config={k: (str(v) if not isinstance(v, (int, float, str, list, dict, bool, type(None))) else v) for k, v in CFG.items()},
    ni_margin=NI_MARGIN, hb_unit_note=hb_unit_note, demo_cols=DEMO_MODEL_COLS, aux_targets=AUX_TARGETS,
    n_features=dict(physics=len(PHYS_COLS), generic=len(GEN_COLS), qc=len(QC_COLS)),
    hardware=dict(cpu=platform.processor(), n_cpu=N_CPU, n_jobs=N_JOBS, gpu=GPU_NAME, torch=HAS_TORCH),
    versions=dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__,
                  sklearn=sklearn.__version__, scipy=scipy.__version__, lightgbm=lgb.__version__, shap=shap.__version__),
    platform=platform.platform())
(TBL_DIR/f"run_manifest_{TAG}.json").write_text(json.dumps(manifest, indent=2, default=str))
print(headline.T.rename(columns={0:"value"}).to_string())
print(f"\n{len(list(TBL_DIR.glob('*.csv')))} tables and {len(list(FIG_DIR.glob('*.pdf')))} vector figures written to {RESULTS_ROOT}")

                                                                 value
n_subjects                                                         252
hb_mean                                                       13.90754
hb_sd                                                           1.4731
mean_baseline_mae                                             1.173788
demographic_only_mae                                          0.814063
demographic_only_r2                                           0.445837
optical_only_mae                                              1.175627
optical_only_r2                                              -0.048912
combined_mae                                                  0.861539
OIG_dR2                                                      -0.063577
OIG_dMAE                                                     -0.047476
OIG_z                                                        -1.832414
OIG_p_perm                                                    0.945274
best_b

### Cell 34 - Substantive integrity gates
The previous notebook's final checks confirmed that fifteen subsets existed, that no predictions were missing and that eight figures had been written - and all of them passed on a run whose model had negative R-squared and zero anaemia sensitivity. A gate suite that cannot fail on a null result gives false assurance. These gates are split into two tiers. **Structural gates** must all pass or the run is invalid. **Scientific gates** are allowed to fail, but each failure explicitly withdraws a class of claim, and the cell prints which claims survive. This is the mechanism that stops a null result from being written up as a wavelength-reduction finding.

In [34]:
structural = {
 "15 unique subsets in ablation": oof_ablation.subset.nunique() == 15,
 "no missing out-of-fold predictions": bool(oof_ablation[["y_true","y_pred"]].notna().all().all()),
 "fold manifest covers all subjects": set(fold_manifest.subject_id) == set(data.subject_id),
 "predictor blocks disjoint": not (set(OPTICAL_ALL) & set(DEMO_MODEL_COLS)) and not (set(OPTICAL_ALL) & set(QC_COLS)),
 "DC preserved in physics bank": bool(all(feat_df[c].median() > 0 for c in dc_cols)),
 "SHAP aligned to surviving features": len(shap_imp) > 0 and set(shap_imp.index).issubset(set(ABL_COLS)),
 "attribution audit spans importance range": len(audit) >= 10,
 "figures and tables written": len(list(FIG_DIR.glob('*.pdf'))) >= 7 and len(list(TBL_DIR.glob('*.csv'))) >= 10}
opt_r2, demo_r2 = get_r2("optical_only"), get_r2("demographic_only")
mean_mae = float(arm_summary.loc[arm_summary.model == "mean_baseline","mae"].iloc[0])
scientific = {
 "optical model beats the mean baseline": opt_r2 > 0 and float(arm_summary.loc[arm_summary.model=="optical_only","mae"].iloc[0]) < mean_mae,
 "calibration slope is plausible (0.2-1.8)": 0.2 <= agr.loc[0,"calib_slope"] <= 1.8,
 "optical information gain is significant": bool(p_oig < 0.05),
 "optical-only outperforms demographic-only": opt_r2 > demo_r2,
 "subset spread exceeds twice the seed noise": bool(np.isfinite(SEED_SD) and SUBSET_SPREAD > 2*SEED_SD),
 "SHAP agrees with LOFO beyond the null": bool(np.isfinite(p_aci) and p_aci < 0.05)}
for label, d in [("STRUCTURAL", structural), ("SCIENTIFIC", scientific)]:
    print(f"\n--- {label} ---")
    for k, v in d.items(): print(f"  {'PASS' if v else 'FAIL'} - {k}")
assert all(structural.values()), "A structural gate failed: this run is invalid, do not report it."
print("\n" + "="*78 + "\nCLAIMS PERMITTED BY THIS RUN\n" + "="*78)
claims = []
claims.append(("A confound-controlled estimate of optical information on this dataset", True))
claims.append(("Optics carry Hb information beyond demographics", scientific["optical information gain is significant"]))
_bs = bank_summary.set_index("bank")["mae"]
claims.append(("Beer-Lambert features outperform generic waveform statistics",
               bool(_bs.get("physics", np.inf) < _bs.get("generic", np.inf) and np.isfinite(p_pg) and p_pg < 0.05)))
claims.append(("Any wavelength can be removed without loss (channel-reduction claim)",
               scientific["optical model beats the mean baseline"] and scientific["subset spread exceeds twice the seed noise"] and bool(paired.non_inferior.any())))
claims.append(("The SHAP wavelength ranking reflects model mechanism", scientific["SHAP agrees with LOFO beyond the null"]))
claims.append(("SHAP rankings are indistinguishable from those of an uninformed model (cautionary result)",
               bool(np.isfinite(p_aci) and p_aci >= 0.05)))
claims.append(("Anaemia screening performance is reportable as more than exploratory", bool(scr["n_positive"] >= 30)))
for text, ok in claims: print(f"  {'SUPPORTED    ' if ok else 'NOT SUPPORTED'} | {text}")
pd.DataFrame(claims, columns=["claim","supported"]).to_csv(TBL_DIR/f"t17_claim_register_{TAG}.csv", index=False)
print("\nThe claim register is written to t17_claim_register. Do not assert in the manuscript any claim marked NOT SUPPORTED.")


--- STRUCTURAL ---
  PASS - 15 unique subsets in ablation
  PASS - no missing out-of-fold predictions
  PASS - fold manifest covers all subjects
  PASS - predictor blocks disjoint
  PASS - DC preserved in physics bank
  PASS - SHAP aligned to surviving features
  PASS - attribution audit spans importance range
  PASS - figures and tables written

--- SCIENTIFIC ---
  FAIL - optical model beats the mean baseline
  FAIL - calibration slope is plausible (0.2-1.8)
  FAIL - optical information gain is significant
  FAIL - optical-only outperforms demographic-only
  PASS - subset spread exceeds twice the seed noise
  FAIL - SHAP agrees with LOFO beyond the null

CLAIMS PERMITTED BY THIS RUN
  SUPPORTED     | A confound-controlled estimate of optical information on this dataset
  NOT SUPPORTED | Optics carry Hb information beyond demographics
  NOT SUPPORTED | Beer-Lambert features outperform generic waveform statistics
  NOT SUPPORTED | Any wavelength can be removed without loss (channel-re

---
## 9. Limitations to carry into the Discussion

- The effective independent sample size is 252 **subjects**, not 1008 segments and not 12,000 samples per channel. Every interval in this notebook is bootstrapped at subject level for that reason.
- This is a public retrospective dataset with no prospective clinical validation and no external cohort. Generalisation to other sensors, skin tones, contact pressures and populations is untested, and the dataset authors themselves flag skin tone and finger thickness as uncontrolled sources of variation.
- The geometry is **reflectance**, not transmittance. Beer-Lambert features derived for transmittance pulse oximetry do not transfer without assumption, and this is an active point of disagreement in the literature rather than settled ground.
- Channel count is a transparent proxy for acquisition complexity. No hardware was built, no power budget was measured, and no monetary cost is claimed.
- Anaemia prevalence in this cohort is low and severe anaemia is essentially absent, so all screening metrics are underpowered and are reported as exploratory with the positive-class count attached.
- SHAP attribution is model-specific and is not causal. The null-calibrated audit in Cell 23 tests whether an attribution is distinguishable from chance; it does not establish that a credible attribution identifies a physical mechanism.
- A non-inferiority result at a 0.2 g/dL margin on this cohort is not clinical equivalence, and confidence intervals do not establish it.
- The 1D-CNN arm is deliberately small. With 252 subjects it is a test of whether a learned representation recovers information the handcrafted banks miss, not a claim about what a well-powered deep model could achieve.

## 10. Running this

1. Leave `SMOKE_TEST = True` and run all cells top to bottom. Expect a few minutes on the target laptop. Every structural gate in Cell 34 must pass.
2. Read the scientific gates. They are expected to fail in smoke mode because of the reduced subject cap; that is not an error.
3. Set `SMOKE_TEST = False`, restart the kernel, and run all again. Do not run cells out of order - later cells depend on names bound earlier.
4. Bump `SCHEMA` in Cell 9 if you change any feature definition. The cache is keyed on it, so stale features cannot silently survive an edit.
5. Preserve `tables/`, `figures/` and the fold manifest from the final run. Only final-mode artefacts should feed the manuscript.
6. Write the paper from `t0_headline_metrics` and `t17_claim_register`. The claim register is the contract: anything marked NOT SUPPORTED does not go in the abstract.